In [1]:
import pandas as pd
import geopandas as gpd
from IPython.display import display
import requests as re
from io import BytesIO
from os import path, makedirs


from core.downloads.geosampa import get_capabilities, get_features
from core.downloads.ceinfo import load_consultas
from core.downloads.orcamento import load_orcamento, load_orcamento_r

# GT Saúde

O Grupo de Trabalho de Saúde definiu os indicadores sem a utilização dos formulários, em reuniões posteriores. Por isso, os indicadores serão descritos diretamente neste notebook para documentação.

A base inicial para os indicadores é o capítulo 7, Eficiência nas macrorregiões de saúde no Sistema Único de Saúde: uma abordagem comparativa – 2008-2017, do livro [SUS: avaliação da eficiência do gasto público em saúde](https://repositorio.ipea.gov.br/handle/11058/12029). Este capítulo busca associar os custos e infraestrutura relacionada ao SUS e o impacto na saúde da população de cada macrorregião, utilizando as seguintes variáveis:

- Taxa de mortalidade por causas evitáveis (100.000 hab.);
- Taxa de mortalidade por causas tratáveis (100.000 hab.);
- Taxa de mortalidade por causas evitáveis/tratáveis (100.000 hab.);
- Despesas per capita em atenção básica e vigilância;
- Despesas per capita em atenção especializada e Suporte profilático terapêutico;
- Leitos de internações/hab. (x100.000);
- Leitos complementares/hab. (x100.000);
- Equipamentos respiradores e ventiladores SUS/hab. (x100.000);
- População (em mil hab.);
- Área (km2);
- Densidade demográfica (hab./km²);
- Idosos %;
- PIB per capita;
- Usuários de planos de saúde %.


A partir dessas referência, o GT avaliou a disponibilidade de dados semelhantes a nível intramunicipal e complementou com informações de produção dos serviços de saúde, chegando na seguinte lista de variáveis, sempre que possível agregando por Subprefeitura:

- Taxa de mortalidade por causas evitáveis (100.000 hab.);
- Taxa de mortalidade por causas tratáveis (100.000 hab.);
- Taxa de mortalidade por causas evitáveis/tratáveis (100.000 hab.);
- Leitos de internações/hab. (x100.000);
- Leitos complementares/hab. (x100.000);
- Leitos de UTI/hab. (x100.000);
- Despesas liquidadas em atenção básica e vigilância;
- Despesas liquidadas em atenção especializada e Suporte profilático terapêutico;
- Consultas na Atenção Básica;
- Consultas na Atenção Especializada;
- Consultas na Urgência/Emergência;
- Número de Unidades Básicas de Saúde;
- Número de Ambulatórios especializados;
- Número de Unidades de Saúde Mental;
- Número de Unidades de Urgência e Emergência;
- Número de Hospitais;
- Número de Unidades DST/AIDS;
- Número de demais unidades;

A partir dessa definição, vamos obter as variáveis.

# Extração das variáveis e transformação inicial

## Número de unidades

Estabelecimentos de saúde municipais, estaduais, federais e privados do município estão disponíveis no GeoSampa, em diferentes camadas. Primeiro, vamos avaliar quais são essas camadas e, depois, baixá-las e agrupá-las da maneira decidida pelo GT.

In [2]:
equipamento_saude_typenames = get_capabilities('equipamento_saude')
equipamento_saude_typenames

[{'name': 'geoportal:equipamento_saude_abrangencia_ubs',
  'title': 'Abrangência das Unidades Básicas de Saúde',
  'abstract': 'Áreas de abrangência das UBS.'},
 {'name': 'geoportal:equipamento_saude_ambulatorios_especializados',
  'title': 'Ambulatórios especializados',
  'abstract': 'Ambulatórios Especializados do cadastro de estabelecimentos de saúde municipais, estaduais, federais e privados da Cidade de São Paulo. Atende pacientes encaminhados por outros estabelecimentos e que requerem cuidados em áreas específicas.'},
 {'name': 'geoportal:equipamento_saude_coordenadoria_regional',
  'title': 'Coordenadoria Regional de Saúde',
  'abstract': 'Limites administrativos da Coordenadoria Regional de Saúde.'},
 {'name': 'geoportal:equipamento_saude_hospital',
  'title': 'Hospital',
  'abstract': 'Hospitais do cadastro de estabelecimentos de saúde municipais, estaduais, federais e privados da Cidade de São Paulo. Estabelecimentos de saúde com capacidade de internamento (leitos) e que real

Primeiro, removemos todas as camadas que não representam os pontos das unidades de saúde. Além disso, a camada de vigilância não retorna nenhuma unidade, então vamos removê-la também.

In [3]:
del equipamento_saude_typenames[0]
del equipamento_saude_typenames[1]
del equipamento_saude_typenames[4]
del equipamento_saude_typenames[8]
del equipamento_saude_typenames[7]

equipamento_saude_typenames

[{'name': 'geoportal:equipamento_saude_ambulatorios_especializados',
  'title': 'Ambulatórios especializados',
  'abstract': 'Ambulatórios Especializados do cadastro de estabelecimentos de saúde municipais, estaduais, federais e privados da Cidade de São Paulo. Atende pacientes encaminhados por outros estabelecimentos e que requerem cuidados em áreas específicas.'},
 {'name': 'geoportal:equipamento_saude_hospital',
  'title': 'Hospital',
  'abstract': 'Hospitais do cadastro de estabelecimentos de saúde municipais, estaduais, federais e privados da Cidade de São Paulo. Estabelecimentos de saúde com capacidade de internamento (leitos) e que realizam procedimentos de média e alta complexidade.'},
 {'name': 'geoportal:equipamento_saude_outros',
  'title': 'Saúde - Outros',
  'abstract': 'Estabelecimentos e serviços especializados do cadastro de estabelecimentos de saúde municipais, estaduais, federais e privados da Cidade de São Paulo.'},
 {'name': 'geoportal:equipamento_saude_saude_mental

In [4]:
gdfs_equip = []

for t in equipamento_saude_typenames:
    gdfs_equip.append(get_features(t.get('name')))

In [5]:
for gdf in gdfs_equip:
    display(gdf.crs)
    display(gdf.head(2))

<Projected CRS: EPSG:31983>
Name: SIRGAS 2000 / UTM zone 23S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 48°W and 42°W, northern and southern hemispheres, onshore and offshore.
- bounds: (-48.0, -33.5, -42.0, 5.13)
Coordinate Operation:
- name: UTM zone 23S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

,id,cd_identificador,cd_identificador_tema_equipamento,nm_tema_equipamento,cd_identificador_rede_equipamento,nm_rede_equipamento,tx_rede_equipamento,cd_identificador_tipo_equipamento,nm_tipo_equipamento,tx_tipo_equipamento,...,tx_esfera_administrativa_equipamento,cd_equipamento,nm_equipamento,tx_endereco_equipamento,nm_bairro_equipamento,cd_cep_equipamento,tx_horario_funcionamento,tx_numero_telefone,cd_equipamento_base_origem,geometry
0,equipamento_saude_ambulatorios_especializados....,120001,6,SAUDE,0,SEM REDE,Sem descrição,4,Assistência Médica Ambulatorial,"Atua na área de atenção básica, integrada e ar...",...,Administração municipal,963,AMA 12H BORACEA - DR. LUIZ BACCALA,"R. DR. RIBEIRO DE ALMEIDA, 14",BARRA FUNDA,01137-020,None,4210-5396 / 3392-1944,None,POINT (331397.001 7397422.322)
1,equipamento_saude_ambulatorios_especializados....,120002,6,SAUDE,0,SEM REDE,Sem descrição,4,Assistência Médica Ambulatorial,"Atua na área de atenção básica, integrada e ar...",...,Administração municipal,935,AMA 12H COMPLEXO PRATES,"R. PRATES, 1101",BOM RETIRO,01121-000,None,3229-8192,None,POINT (333135.515 7397559.623)


<Projected CRS: EPSG:31983>
Name: SIRGAS 2000 / UTM zone 23S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 48°W and 42°W, northern and southern hemispheres, onshore and offshore.
- bounds: (-48.0, -33.5, -42.0, 5.13)
Coordinate Operation:
- name: UTM zone 23S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

,id,cd_identificador,cd_identificador_tema_equipamento,nm_tema_equipamento,cd_identificador_rede_equipamento,nm_rede_equipamento,tx_rede_equipamento,cd_identificador_tipo_equipamento,nm_tipo_equipamento,tx_tipo_equipamento,...,tx_esfera_administrativa_equipamento,cd_equipamento,nm_equipamento,tx_endereco_equipamento,nm_bairro_equipamento,cd_cep_equipamento,tx_horario_funcionamento,tx_numero_telefone,cd_equipamento_base_origem,geometry
0,equipamento_saude_hospital.120093,120093,6,SAUDE,0,SEM REDE,Sem descrição,32,Hospital Especializado,É o estabelecimento de saúde destinado a prest...,...,Administração estadual vinculada à Secretaria ...,959,AMB ESPEC DR. GERALDO PAULO BOURROL (CONSOLAÇÃO),"R. MARTINS FONTES, 208",CENTRO,01050-000,None,2141-6200,None,POINT (332182.763 7394765.185)
1,equipamento_saude_hospital.120108,120108,6,SAUDE,0,SEM REDE,Sem descrição,32,Hospital Especializado,É o estabelecimento de saúde destinado a prest...,...,Administração estadual vinculada à Secretaria ...,366,AME DR. LUIZ ROBERTO BARRADAS BARATA,"AV. ALMIRANTE DELAMARE, 1534",CIDADE NOVA HELIOPOLIS,04230-000,None,2065-1550,None,POINT (337785.173 7388015.086)


<Projected CRS: EPSG:31983>
Name: SIRGAS 2000 / UTM zone 23S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 48°W and 42°W, northern and southern hemispheres, onshore and offshore.
- bounds: (-48.0, -33.5, -42.0, 5.13)
Coordinate Operation:
- name: UTM zone 23S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

...

,id,cd_identificador,cd_identificador_tema_equipamento,nm_tema_equipamento,cd_identificador_rede_equipamento,nm_rede_equipamento,tx_rede_equipamento,cd_identificador_tipo_equipamento,nm_tipo_equipamento,tx_tipo_equipamento,...,tx_esfera_administrativa_equipamento,cd_equipamento,nm_equipamento,tx_endereco_equipamento,nm_bairro_equipamento,cd_cep_equipamento,tx_horario_funcionamento,tx_numero_telefone,cd_equipamento_base_origem,geometry
0,equipamento_saude_ubs_posto_centro.120031,120031,6,SAUDE,0,SEM REDE,Sem descrição,45,Unidade Básica de Saúde,"Atendimento básico em Pediatria, Ginecologia, ...",...,Administração municipal,580,AMA/UBS ÁGUA RASA - DR. MARCOS ANDRADE CORSATO,"R. SERRA DE JAIRE, 1480",ÁGUA RASA,03175-001,None,2605-5307,None,POINT (338714.702 7394301.385)
1,equipamento_saude_ubs_posto_centro.120032,120032,6,SAUDE,0,SEM REDE,Sem descrição,45,Unidade Básica de Saúde,"Atendimento básico em Pediatria, Ginecologia, ...",...,Administração municipal,429,AMA/UBS AGUIA DE HAIA,"R. TANTAS PALAVRAS, 59",COHAB ÁGUIA DE HAIA,08223-510,None,3756-3264,None,POINT (350002.697 7397919.976)


<Projected CRS: EPSG:31983>
Name: SIRGAS 2000 / UTM zone 23S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 48°W and 42°W, northern and southern hemispheres, onshore and offshore.
- bounds: (-48.0, -33.5, -42.0, 5.13)
Coordinate Operation:
- name: UTM zone 23S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

,id,cd_identificador,cd_identificador_tema_equipamento,nm_tema_equipamento,cd_identificador_rede_equipamento,nm_rede_equipamento,tx_rede_equipamento,cd_identificador_tipo_equipamento,nm_tipo_equipamento,tx_tipo_equipamento,...,tx_esfera_administrativa_equipamento,cd_equipamento,nm_equipamento,tx_endereco_equipamento,nm_bairro_equipamento,cd_cep_equipamento,tx_horario_funcionamento,tx_numero_telefone,cd_equipamento_base_origem,geometry
0,equipamento_saude_unidades_dst-aids.120363,120363,6,SAUDE,0,SEM REDE,Sem descrição,19,Centro de Testagem e Acompanhamento DST/AIDS,Serviço estruturado para atividades de prevenç...,...,Administração municipal,224,CTA IST/AIDS CIDADE TIRADENTES,"R. MILAGRE DOS PEIXES, 357",CONJUNTO HABITACIONA,08474-120,None,2282-7055 / 2964-0784,None,POINT (356240.559 7391275.466)
1,equipamento_saude_unidades_dst-aids.120364,120364,6,SAUDE,0,SEM REDE,Sem descrição,19,Centro de Testagem e Acompanhamento DST/AIDS,Serviço estruturado para atividades de prevenç...,...,Administração municipal,319,CTA IST/AIDS GUAIANASES,"R. CENTRALINA, 168",GUAIANASES,08410-000,None,2554-5312,None,POINT (355449.169 7395443.615)


<Projected CRS: EPSG:31983>
Name: SIRGAS 2000 / UTM zone 23S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Brazil - between 48°W and 42°W, northern and southern hemispheres, onshore and offshore.
- bounds: (-48.0, -33.5, -42.0, 5.13)
Coordinate Operation:
- name: UTM zone 23S
- method: Transverse Mercator
Datum: Sistema de Referencia Geocentrico para las AmericaS 2000
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

,id,cd_identificador,cd_identificador_tema_equipamento,nm_tema_equipamento,cd_identificador_rede_equipamento,nm_rede_equipamento,tx_rede_equipamento,cd_identificador_tipo_equipamento,nm_tipo_equipamento,tx_tipo_equipamento,...,tx_esfera_administrativa_equipamento,cd_equipamento,nm_equipamento,tx_endereco_equipamento,nm_bairro_equipamento,cd_cep_equipamento,tx_horario_funcionamento,tx_numero_telefone,cd_equipamento_base_origem,geometry
0,equipamento_saude_urgencia_emergencia.120007,120007,6,SAUDE,0,SEM REDE,Sem descrição,4,Assistência Médica Ambulatorial,"Atua na área de atenção básica, integrada e ar...",...,Administração municipal,78,AMA 24H CAPÃO REDONDO,"AV. COMENDADOR SANTANA, 774",JD BOA ESPERANÇA,05866-000,None,5872-9917 / 5872-9919,None,POINT (319131.694 7380616.296)
1,equipamento_saude_urgencia_emergencia.120008,120008,6,SAUDE,0,SEM REDE,Sem descrição,4,Assistência Médica Ambulatorial,"Atua na área de atenção básica, integrada e ar...",...,Administração municipal,646,AMA 24H ENGENHEIRO GOULART - DR. JOSÉ PIRES,"R. AUGUSTO CORREIA LEITE, 538",ENG. GOULART,03726-100,None,2038-1626 / 2622-0366,None,POINT (345081.544 7400275.725)


In [6]:
gdf_equip = pd.concat(gdfs_equip)
gdf_equip = gpd.GeoDataFrame(gdf_equip)
gdf_equip = gdf_equip.set_geometry(gdfs_equip[0].geometry.name)
gdf_equip = gdf_equip.set_crs(gdfs_equip[0].crs)
gdf_equip

,id,cd_identificador,cd_identificador_tema_equipamento,nm_tema_equipamento,cd_identificador_rede_equipamento,nm_rede_equipamento,tx_rede_equipamento,cd_identificador_tipo_equipamento,nm_tipo_equipamento,tx_tipo_equipamento,...,tx_esfera_administrativa_equipamento,cd_equipamento,nm_equipamento,tx_endereco_equipamento,nm_bairro_equipamento,cd_cep_equipamento,tx_horario_funcionamento,tx_numero_telefone,cd_equipamento_base_origem,geometry
0,equipamento_saude_ambulatorios_especializados....,120001,6,SAUDE,0,SEM REDE,Sem descrição,4,Assistência Médica Ambulatorial,"Atua na área de atenção básica, integrada e ar...",...,Administração municipal,963,AMA 12H BORACEA - DR. LUIZ BACCALA,"R. DR. RIBEIRO DE ALMEIDA, 14",BARRA FUNDA,01137-020,None,4210-5396 / 3392-1944,None,POINT (331397.001 7397422.322)
1,equipamento_saude_ambulatorios_especializados....,120002,6,SAUDE,0,SEM REDE,Sem descrição,4,Assistência Médica Ambulatorial,"Atua na área de atenção básica, integrada e ar...",...,Administração municipal,935,AMA 12H COMPLEXO PRATES,"R. PRATES, 1101",BOM RETIRO,01121-000,None,3229-8192,None,POINT (333135.515 7397559.623)
2,equipamento_saude_ambulatorios_especializados....,120003,6,SAUDE,0,SEM REDE,Sem descrição,4,Assistência Médica Ambulatorial,"Atua na área de atenção básica, integrada e ar...",...,Administração municipal,17,AMA 12H JARDIM PERI PERI,"R. JOÃO GUERRA, 247",JD PERI PERI,05535-100,None,3744-3021 / 3744-3082,None,POINT (323248.398 7390472.108)
3,equipamento_saude_ambulatorios_especializados....,120004,6,SAUDE,0,SEM REDE,Sem descrição,4,Assistência Médica Ambulatorial,"Atua na área de atenção básica, integrada e ar...",...,Administração municipal,546,AMA 12H JARDIM SÃO LUIZ,"R. LUIZ ANTONIO VERNEY, 98",JD SÃO LUIZ,05846-380,None,5511-1153 / 5816-5539,None,POINT (322162.36 7382246.957)
4,equipamento_saude_ambulatorios_especializados....,120005,6,SAUDE,0,SEM REDE,Sem descrição,4,Assistência Médica Ambulatorial,"Atua na área de atenção básica, integrada e ar...",...,Administração municipal,658,AMA 12H MAURICE PATE,"R. FREI GERMANO, 50",PENHA,03604-050,None,2097-5802,None,POINT (341684.865 7397317.38)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48,equipamento_saude_urgencia_emergencia.121025,121025,6,SAUDE,0,SEM REDE,Sem descrição,38,Pronto Atendimento,Atendimento de urgência dentro do horário de f...,...,Administração municipal,543,UPA VERA CRUZ,"AV. DOS FUNCIONARIOS PUBLICOS, 379",JARDIM VERA CRUZ,04962-000,None,5898-5900,None,POINT (318482.017 7373748.846)
49,equipamento_saude_urgencia_emergencia.121026,121026,6,SAUDE,0,SEM REDE,Sem descrição,38,Pronto Atendimento,Atendimento de urgência dentro do horário de f...,...,Administração municipal,952,UPA VERGUEIRO,"R. VERGUEIRO, 613",LIBERDADE,01504-001,None,4040-1067,None,POINT (332661.412 7392701.246)
50,equipamento_saude_urgencia_emergencia.121027,121027,6,SAUDE,0,SEM REDE,Sem descrição,38,Pronto Atendimento,Atendimento de urgência dentro do horário de f...,...,Administração municipal,1008,UPA VILA MARIA - DR JOSE MAURO DEL ROIO CORREA,"PRAÇA ENGENHEIRO HUGO BRANDI, 15",VILA MARIA BAIXA,02168-050,None,3500-2560,None,POINT (338962.421 7398328.416)
51,equipamento_saude_urgencia_emergencia.121028,121028,6,SAUDE,0,SEM REDE,Sem descrição,38,Pronto Atendimento,Atendimento de urgência dentro do horário de f...,...,Administração municipal,1034,UPA VILA MARIANA,"R. DIOGO DE FARIA, 609",V CLEMENTINO,04037-002,None,3245-9010,None,POINT (332386.864 7389557.568)


## Consultas médicas

Os dados de consultas médicas serão obtidos do boletim "Saúde em dados", disponibilizado anualmente pela CEINFO, da Secretaria Municipal de Saúde. Existe uma versão do boletim em excel, regionalizado por subprefeituras. Primeiro, vamos confirmar as planilhas disponíveis no arquivo.

In [7]:
ANOS_CONSULTAS = [2024, 2025]

df_consultas_ajustado = pd.concat([load_consultas(year=ano)
                                   for ano in ANOS_CONSULTAS])

## Leitos de internação e complementares

Os dados de leitos não estão disponíveis de maneira regionalizada no boletim "Saúde em dados", então terão de ser obtidos de outra fonte. Eles estão disponíveis na plataforma [TabNet](http://tabnet.saude.prefeitura.sp.gov.br/cgi/deftohtm3.exe?secretarias/saude/TABNET/cnes/leito.def). Entretanto, os links dos arquivos gerados nessa plataforma podem não ser permanentes, então utilizaremos uma versão já baixada do arquivo csv.

Obs: caso a necessidade fosse mais geral ou a atualização mais frequente, seria possível desenvolver um *scrapper* para a plataforma, mas estrapola o escopo deste trabalho.

In [8]:
data_path = path.join('data', 'input', 'saude')
leitos_filename = 'A161721192_29_142_217.csv'
leitos_file_path = path.join(data_path, leitos_filename)

df_leito = pd.read_csv(leitos_file_path,
                        skiprows = 3,
                        sep = ';',
                        encoding = 'latin1')
df_leito

,Pref Regional,1-Cirúrgico,2-Clínico,3-Complementar,4-Obstétrico,5-Pediátrico,6-Outras Especialidades,7-Hospital/DIA,Total
0,Aricanduva/ Formosa/ Carrão,-,-,-,-,-,95,-,95.0
1,Butantã,78,131,66,57,70,8,30,440.0
2,Campo Limpo,-,-,-,-,-,88,7,95.0
3,Capela do Socorro,78,155,96,62,38,28,-,457.0
4,Casa Verde/ Cachoeirinha,86,74,104,86,33,-,4,387.0
5,Cidade Ademar,-,-,-,-,-,-,14,14.0
6,Cidade Tiradentes,49,54,45,38,27,10,1,224.0
7,Ermelino Matarazzo,85,58,44,40,29,-,-,256.0
8,Freguesia/ Brasilândia,107,138,85,-,28,16,13,387.0
9,Guaianases,40,58,46,48,33,24,-,249.0


Primeiro, vamos excluir as linhas sobressalentes ao final do dataframe e a coluna de totalização.

In [9]:
df_leito = df_leito.iloc[:32]
df_leito

,Pref Regional,1-Cirúrgico,2-Clínico,3-Complementar,4-Obstétrico,5-Pediátrico,6-Outras Especialidades,7-Hospital/DIA,Total
0,Aricanduva/ Formosa/ Carrão,-,-,-,-,-,95,-,95.0
1,Butantã,78,131,66,57,70,8,30,440.0
2,Campo Limpo,-,-,-,-,-,88,7,95.0
3,Capela do Socorro,78,155,96,62,38,28,-,457.0
4,Casa Verde/ Cachoeirinha,86,74,104,86,33,-,4,387.0
5,Cidade Ademar,-,-,-,-,-,-,14,14.0
6,Cidade Tiradentes,49,54,45,38,27,10,1,224.0
7,Ermelino Matarazzo,85,58,44,40,29,-,-,256.0
8,Freguesia/ Brasilândia,107,138,85,-,28,16,13,387.0
9,Guaianases,40,58,46,48,33,24,-,249.0


In [10]:
cols_total = df_leito.columns.str.lower().str.contains('total')
df_leito = df_leito.loc[:,~cols_total]
df_leito

,Pref Regional,1-Cirúrgico,2-Clínico,3-Complementar,4-Obstétrico,5-Pediátrico,6-Outras Especialidades,7-Hospital/DIA
0,Aricanduva/ Formosa/ Carrão,-,-,-,-,-,95,-
1,Butantã,78,131,66,57,70,8,30
2,Campo Limpo,-,-,-,-,-,88,7
3,Capela do Socorro,78,155,96,62,38,28,-
4,Casa Verde/ Cachoeirinha,86,74,104,86,33,-,4
5,Cidade Ademar,-,-,-,-,-,-,14
6,Cidade Tiradentes,49,54,45,38,27,10,1
7,Ermelino Matarazzo,85,58,44,40,29,-,-
8,Freguesia/ Brasilândia,107,138,85,-,28,16,13
9,Guaianases,40,58,46,48,33,24,-


Em seguida, "despivotamos" a tabela.

In [11]:
df_leito_ajustado = df_leito.melt(id_vars=['Pref Regional'],
                                    var_name='Categoria específica',
                                    value_name='Qtd leitos')
df_leito_ajustado

,Pref Regional,Categoria específica,Qtd leitos
0,Aricanduva/ Formosa/ Carrão,1-Cirúrgico,-
1,Butantã,1-Cirúrgico,78
2,Campo Limpo,1-Cirúrgico,-
3,Capela do Socorro,1-Cirúrgico,78
4,Casa Verde/ Cachoeirinha,1-Cirúrgico,86
...,...,...,...
219,Sapopemba,7-Hospital/DIA,6
220,Sé,7-Hospital/DIA,69
221,Vila Maria/ Vila Guilherme,7-Hospital/DIA,6
222,Vila Mariana,7-Hospital/DIA,166


Por último, substituimos os `-` por 0.

In [12]:
hifen = df_leito_ajustado['Qtd leitos'].str.contains('-')
df_leito_ajustado.loc[hifen,'Qtd leitos'] = 0
df_leito_ajustado.loc[:,'Qtd leitos'] = df_leito_ajustado.loc[:,'Qtd leitos'].astype(int)
df_leito_ajustado

,Pref Regional,Categoria específica,Qtd leitos
0,Aricanduva/ Formosa/ Carrão,1-Cirúrgico,0
1,Butantã,1-Cirúrgico,78
2,Campo Limpo,1-Cirúrgico,0
3,Capela do Socorro,1-Cirúrgico,78
4,Casa Verde/ Cachoeirinha,1-Cirúrgico,86
...,...,...,...
219,Sapopemba,7-Hospital/DIA,6
220,Sé,7-Hospital/DIA,69
221,Vila Maria/ Vila Guilherme,7-Hospital/DIA,6
222,Vila Mariana,7-Hospital/DIA,166


Posteriormente, cada categoria específica dos leitos será categorizada em Leitos de internações, leitos complementares ou leitos de UTI, de acordo com as variáveis definidas.

## Mortalidade

Os dados sobre mortalidade também serão extraídos da plataforma [TabNet](http://tabnet.saude.prefeitura.sp.gov.br/cgi/deftohtm3.exe?secretarias/saude/TABNET/cnes/leito.def). Novamente, como os links dos arquivos gerados nessa plataforma podem não ser permanentes, utilizaremos uma versão já baixada do arquivo csv.

In [13]:
data_path = path.join('data', 'input', 'saude')
mortalidade_filename = 'A171717192_29_141_228.csv'
mortalidade_file_path = path.join(data_path, mortalidade_filename)

df_mortalidade = pd.read_csv(mortalidade_file_path,
                        skiprows = 4,
                        sep = ';',
                        encoding = 'latin1')
df_mortalidade

,Subprefeitura residência,Doencas infec intest,Tuberculose,Infec menigocóccica,Septicemia,Aids,Covid19,Hepatites virais,CA esôfago,CA estômago,...,Mal definidas,Acid trânsito e transporte terrestres,Quedas acidentais,Demais acidentes,Suicídios,Homicídios,Lesões intenc indeterminada,Demais causas externas,Demais causas de morte,Total
0,Aricanduva/Formosa/Carrão,4,9.0,-,9.0,6.0,2,3,3,15.0,...,38.0,10.0,5,19.0,3,3,46.0,1,260.0,1177.0
1,Butantã,5,10.0,1,12.0,12.0,3,2,8,20.0,...,62.0,13.0,10,23.0,6,10,66.0,3,292.0,1380.0
2,Campo Limpo,7,16.0,-,17.0,23.0,7,7,19,34.0,...,119.0,30.0,15,22.0,9,18,104.0,8,459.0,2254.0
3,Capela do Socorro,7,16.0,-,24.0,15.0,3,3,20,35.0,...,99.0,44.0,25,37.0,15,35,80.0,6,509.0,2464.0
4,Casa Verde/Cachoeirinha,4,15.0,-,27.0,19.0,1,3,5,19.0,...,75.0,14.0,5,11.0,5,5,52.0,3,288.0,1438.0
5,Cidade Ademar,-,20.0,2,13.0,14.0,3,1,14,26.0,...,78.0,12.0,8,18.0,14,9,75.0,3,312.0,1650.0
6,Cidade Tiradentes,1,12.0,1,14.0,9.0,1,2,4,7.0,...,45.0,2.0,1,14.0,3,6,51.0,1,183.0,921.0
7,Ermelino Matarazzo,3,10.0,-,12.0,7.0,-,1,5,14.0,...,30.0,4.0,2,16.0,6,3,36.0,-,200.0,906.0
8,Freguesia/Brasilândia,6,20.0,-,23.0,16.0,2,3,11,25.0,...,97.0,19.0,13,26.0,5,8,80.0,2,353.0,1872.0
9,Guaianases,1,16.0,-,16.0,12.0,1,1,4,13.0,...,55.0,2.0,4,10.0,-,5,68.0,3,255.0,1217.0


Primeiro, vamos excluir as linhas sobressalentes ao final do dataframe e a coluna de totalização.

In [14]:
df_mortalidade = df_mortalidade.iloc[:32]
df_mortalidade

,Subprefeitura residência,Doencas infec intest,Tuberculose,Infec menigocóccica,Septicemia,Aids,Covid19,Hepatites virais,CA esôfago,CA estômago,...,Mal definidas,Acid trânsito e transporte terrestres,Quedas acidentais,Demais acidentes,Suicídios,Homicídios,Lesões intenc indeterminada,Demais causas externas,Demais causas de morte,Total
0,Aricanduva/Formosa/Carrão,4,9.0,-,9.0,6.0,2,3,3,15.0,...,38.0,10.0,5,19.0,3,3,46.0,1,260.0,1177.0
1,Butantã,5,10.0,1,12.0,12.0,3,2,8,20.0,...,62.0,13.0,10,23.0,6,10,66.0,3,292.0,1380.0
2,Campo Limpo,7,16.0,-,17.0,23.0,7,7,19,34.0,...,119.0,30.0,15,22.0,9,18,104.0,8,459.0,2254.0
3,Capela do Socorro,7,16.0,-,24.0,15.0,3,3,20,35.0,...,99.0,44.0,25,37.0,15,35,80.0,6,509.0,2464.0
4,Casa Verde/Cachoeirinha,4,15.0,-,27.0,19.0,1,3,5,19.0,...,75.0,14.0,5,11.0,5,5,52.0,3,288.0,1438.0
5,Cidade Ademar,-,20.0,2,13.0,14.0,3,1,14,26.0,...,78.0,12.0,8,18.0,14,9,75.0,3,312.0,1650.0
6,Cidade Tiradentes,1,12.0,1,14.0,9.0,1,2,4,7.0,...,45.0,2.0,1,14.0,3,6,51.0,1,183.0,921.0
7,Ermelino Matarazzo,3,10.0,-,12.0,7.0,-,1,5,14.0,...,30.0,4.0,2,16.0,6,3,36.0,-,200.0,906.0
8,Freguesia/Brasilândia,6,20.0,-,23.0,16.0,2,3,11,25.0,...,97.0,19.0,13,26.0,5,8,80.0,2,353.0,1872.0
9,Guaianases,1,16.0,-,16.0,12.0,1,1,4,13.0,...,55.0,2.0,4,10.0,-,5,68.0,3,255.0,1217.0


In [15]:
cols_total = df_mortalidade.columns.str.lower().str.contains('total')
df_mortalidade = df_mortalidade.loc[:,~cols_total]
df_mortalidade

,Subprefeitura residência,Doencas infec intest,Tuberculose,Infec menigocóccica,Septicemia,Aids,Covid19,Hepatites virais,CA esôfago,CA estômago,...,Demais anomalias congênitas,Mal definidas,Acid trânsito e transporte terrestres,Quedas acidentais,Demais acidentes,Suicídios,Homicídios,Lesões intenc indeterminada,Demais causas externas,Demais causas de morte
0,Aricanduva/Formosa/Carrão,4,9.0,-,9.0,6.0,2,3,3,15.0,...,5,38.0,10.0,5,19.0,3,3,46.0,1,260.0
1,Butantã,5,10.0,1,12.0,12.0,3,2,8,20.0,...,4,62.0,13.0,10,23.0,6,10,66.0,3,292.0
2,Campo Limpo,7,16.0,-,17.0,23.0,7,7,19,34.0,...,12,119.0,30.0,15,22.0,9,18,104.0,8,459.0
3,Capela do Socorro,7,16.0,-,24.0,15.0,3,3,20,35.0,...,18,99.0,44.0,25,37.0,15,35,80.0,6,509.0
4,Casa Verde/Cachoeirinha,4,15.0,-,27.0,19.0,1,3,5,19.0,...,5,75.0,14.0,5,11.0,5,5,52.0,3,288.0
5,Cidade Ademar,-,20.0,2,13.0,14.0,3,1,14,26.0,...,9,78.0,12.0,8,18.0,14,9,75.0,3,312.0
6,Cidade Tiradentes,1,12.0,1,14.0,9.0,1,2,4,7.0,...,4,45.0,2.0,1,14.0,3,6,51.0,1,183.0
7,Ermelino Matarazzo,3,10.0,-,12.0,7.0,-,1,5,14.0,...,4,30.0,4.0,2,16.0,6,3,36.0,-,200.0
8,Freguesia/Brasilândia,6,20.0,-,23.0,16.0,2,3,11,25.0,...,7,97.0,19.0,13,26.0,5,8,80.0,2,353.0
9,Guaianases,1,16.0,-,16.0,12.0,1,1,4,13.0,...,2,55.0,2.0,4,10.0,-,5,68.0,3,255.0


Em seguida, "despivotamos" a tabela.

In [16]:
df_mortalidade_ajustado = df_mortalidade.melt(id_vars=['Subprefeitura residência'],
                                    var_name='Causa específica',
                                    value_name='Qtd óbitos')
df_mortalidade_ajustado

,Subprefeitura residência,Causa específica,Qtd óbitos
0,Aricanduva/Formosa/Carrão,Doencas infec intest,4
1,Butantã,Doencas infec intest,5
2,Campo Limpo,Doencas infec intest,7
3,Capela do Socorro,Doencas infec intest,7
4,Casa Verde/Cachoeirinha,Doencas infec intest,4
...,...,...,...
2139,Sapopemba,Demais causas de morte,279.0
2140,Sé,Demais causas de morte,285.0
2141,Vila Maria/Vila Guilherme,Demais causas de morte,279.0
2142,Vila Mariana,Demais causas de morte,185.0


Por último, substituimos os `-` por 0.

In [17]:
hifen = df_mortalidade_ajustado['Qtd óbitos'].astype(str).str.contains('-')
df_mortalidade_ajustado.loc[hifen,'Qtd óbitos'] = 0
df_mortalidade_ajustado.loc[:,'Qtd óbitos'] = df_mortalidade_ajustado.loc[:,'Qtd óbitos'].astype(int)
df_mortalidade_ajustado

,Subprefeitura residência,Causa específica,Qtd óbitos
0,Aricanduva/Formosa/Carrão,Doencas infec intest,4
1,Butantã,Doencas infec intest,5
2,Campo Limpo,Doencas infec intest,7
3,Capela do Socorro,Doencas infec intest,7
4,Casa Verde/Cachoeirinha,Doencas infec intest,4
...,...,...,...
2139,Sapopemba,Demais causas de morte,279
2140,Sé,Demais causas de morte,285
2141,Vila Maria/Vila Guilherme,Demais causas de morte,279
2142,Vila Mariana,Demais causas de morte,185


Posteriormente, cada causa específica dos óbitos será categorizado em Evitável, Tratável ou Evitável/Tratável, de acordo com uma tabela auxiliar definida pelo GT.

## Despesas

Para que seja possível relacionar as despesas com os outros indicadores, podemos usar os dados de regionalização do orçamento, disponíveis de acordo com o valor de detalhamento da ação. E para manter a completude dos dados, podemos complementar os dados regionalizados com os dados de execução não regionalizados.

Esses dados ficam disponíveis na [página de prestação de contas públicas da Secretaria Municipal da Fazenda](https://orcamento.sf.prefeitura.sp.gov.br/orcamento/execucao.php), na Base de Dados da Regionalização da Execução Orçamentária.

In [18]:
ANOS = [2023, 2024, 2025]

df_despesa_r = pd.concat(
    [load_orcamento_r(ano).assign(ANO=str(ano)) for ano in ANOS],
    ignore_index=True
)
df_despesa_r

,COD_EMPRESA_PMSP,COD_EMPENHO,ANO_EMPENHO,CÓDIGO_NLP,ANO_LIQUIDAÇÃO,DATA_LIQUIDAÇÃO,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_UNIDADE,...,CÓDIGO_EXERCÍCIO_FONTE,CÓDIGO_DESTINAÇÃO_RECURSO,CÓDIGO_VÍNCULO_PMSP,CÓDIGO_TIPO_CRÉDITO_ORÇAMENTÁRIO,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO,ANO
0,01,2,2023,21838,2023,2023-02-06 00:00:00.0000000,36,SMPED,Secretaria Municipal da Pessoa com Deficiência,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,7980,2023
1,01,3,2023,55926,2023,2023-03-16 00:00:00.0000000,16,SME,Secretaria Municipal de Educação,23,...,1,500,9001,0,Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,32900,2023
2,01,5,2023,137689,2023,2023-06-19 00:00:00.0000000,36,SMPED,Secretaria Municipal da Pessoa com Deficiência,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,"19636,65",2023
3,01,5,2023,137717,2023,2023-06-19 00:00:00.0000000,36,SMPED,Secretaria Municipal da Pessoa com Deficiência,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,5271,2023
4,01,5,2023,172700,2023,2023-07-25 00:00:00.0000000,36,SMPED,Secretaria Municipal da Pessoa com Deficiência,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,"35412,03",2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1590168,01,97678,2025,313572,2025,12/04/2025 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,3900,2025
1590169,01,97678,2025,315486,2025,12/05/2025 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Leste,Subprefeitura Ermelino Matarazzo,Supra-Distrital,Despesa Regionalizável,5550,2025
1590170,01,97678,2025,322291,2025,12/13/2025 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Sul,Subprefeitura Jabaquara,Supra-Distrital,Despesa Regionalizável,1600,2025
1590171,01,97678,2025,322437,2025,12/13/2025 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Sul,Subprefeitura Campo Limpo,Supra-Distrital,Despesa Regionalizável,3100,2025


Primeiro, vamos filtrar os dados para manter apenas a função Saúde do orçamento.

In [19]:
df_despesa_r = df_despesa_r[df_despesa_r['DESCRIÇÃO_FUNÇÃO']=='Saúde']
df_despesa_r = df_despesa_r[df_despesa_r['ANO_LIQUIDAÇÃO']==df_despesa_r['ANO']]
df_despesa_r

,COD_EMPRESA_PMSP,COD_EMPENHO,ANO_EMPENHO,CÓDIGO_NLP,ANO_LIQUIDAÇÃO,DATA_LIQUIDAÇÃO,CÓDIGO_ÓRGÃO,SIGLA_ÓRGÃO,DESCRIÇÃO_ÓRGÃO,CÓDIGO_UNIDADE,...,CÓDIGO_EXERCÍCIO_FONTE,CÓDIGO_DESTINAÇÃO_RECURSO,CÓDIGO_VÍNCULO_PMSP,CÓDIGO_TIPO_CRÉDITO_ORÇAMENTÁRIO,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,VALOR_DETALHAMENTO_AÇÃO,ANO
1286,01,138,2023,44535,2023,2023-03-02 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,"357253,26",2023
1287,01,138,2023,64466,2023,2023-03-24 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,"351520,15",2023
1288,01,138,2023,105742,2023,2023-05-10 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Supra-Regional,Supra Subprefeitura,Supra-Distrital,Despesa Não-Regionalizável,"359014,44",2023
1302,01,140,2023,7780,2023,2023-01-19 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Norte,Subprefeitura Freguesia/Brasilândia,Supra-Distrital,Despesa Regionalizável,"6182101,64",2023
1303,01,140,2023,18932,2023,2023-02-01 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Norte,Subprefeitura Freguesia/Brasilândia,Supra-Distrital,Despesa Regionalizável,"12776343,38",2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1590168,01,97678,2025,313572,2025,12/04/2025 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,3900,2025
1590169,01,97678,2025,315486,2025,12/05/2025 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Leste,Subprefeitura Ermelino Matarazzo,Supra-Distrital,Despesa Regionalizável,5550,2025
1590170,01,97678,2025,322291,2025,12/13/2025 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Sul,Subprefeitura Jabaquara,Supra-Distrital,Despesa Regionalizável,1600,2025
1590171,01,97678,2025,322437,2025,12/13/2025 00:00:00.0000000,84,FMS/SMS,Fundo Municipal de Saúde,10,...,1,500,9001,0,Sul,Subprefeitura Campo Limpo,Supra-Distrital,Despesa Regionalizável,3100,2025


Agora, definimos os tipos de dados das colunas numéricas e agrupamos os dados pelas colunas que serão mantidas nos dados finais.

In [20]:
group_cols = ['CÓDIGO_SUBFUNÇÃO',
        'DESCRIÇÃO_SUBFUNÇÃO',
        'CÓDIGO_PROGRAMA',
        'DESCRIÇÃO_PROGRAMA',
        'REGIÃO',
        'SUBPREFEITURA',
        'DISTRITO',
        'TIPO_REGIONALIZAÇÃO',
        'ANO']

measure_cols = ['VALOR_DETALHAMENTO_AÇÃO']

df_despesa_r.loc[:, measure_cols] = df_despesa_r.loc[:, measure_cols].apply(lambda x: x.str.replace('.', ''))
df_despesa_r.loc[:, measure_cols] = df_despesa_r.loc[:, measure_cols].apply(lambda x: x.str.replace(',', '.'))
df_despesa_r.loc[:, measure_cols] = df_despesa_r.loc[:, measure_cols].apply(pd.to_numeric)

df_despesa_r_agrupado = df_despesa_r.groupby(group_cols).sum().round(2).reset_index().loc[:, group_cols + measure_cols]
df_despesa_r_agrupado

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,VALOR_DETALHAMENTO_AÇÃO
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,599462.69
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,46700.58
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,22676.79
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,24147.12
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,863.79
...,...,...,...,...,...,...,...,...,...,...
434,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,Oeste,Subprefeitura Pinheiros,Supra-Distrital,Despesa Regionalizável,2023,3244937.22
435,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,Sul,Subprefeitura M'Boi Mirim,Supra-Distrital,Despesa Regionalizável,2025,423326.81
436,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2023,550000.0
437,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2025,13227383.38


Vamos ver quais valores aparecem como subfunção.

In [21]:
df_despesa_r_agrupado['DESCRIÇÃO_PROGRAMA'].value_counts()

DESCRIÇÃO_PROGRAMA
Ações e Serviços da Saúde em Atenção Básica, Especialidades e Vigilância      253
Ações e Serviços da Saúde em Atenção Hospitalar e de Urgência e Emergência    121
Suporte Administrativo                                                         28
Ações e Serviços da Saúde Animal                                               18
Modernização Tecnológica, Desburocratização e Inovação do Serviço Público       9
Participação, Transparência e Controle Social da Administração Pública          5
Benefícios e Previdência de Funcionários                                        5
Name: count, dtype: int64

Para os indicadores de "Despesas liquidadas em atenção básica e vigilância" e "Despesas liquidadas em atenção especializada e Suporte profilático terapêutico", parece que os programas serão o suficiente, com alguns não se encaixando em nenhuma das categorias.

Vamos agora obter os dados de despesas totais.

In [22]:
df_despesa = pd.concat(
    [load_orcamento(ano).assign(ANO=str(ano)) for ano in ANOS],
    ignore_index=True
)
df_despesa

,DataInicial,DataFinal,Cd_Exercicio,Cd_AnoExecucao,Cd_Dotacao_Id,Administracao,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Unidade,...,Vl_CongeladoLiquido,Disponivel,Vl_ReservadoLiquido,Vl_EmpenhadoLiquido,Vl_Liquidado,Vl_Pago,Saldo_Dotacao,DataExtracao,ANO,ProjAtiv_Descricao
0,01/01/2023,31/12/2023,2023,2023,161750,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,0,0,0,0,0,0,23/01/2024,2023,NaN
1,01/01/2023,31/12/2023,2023,2023,161749,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,0,0,0,0,0,0,23/01/2024,2023,NaN
2,01/01/2023,31/12/2023,2023,2023,159818,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,0,0,0,0,0,0,23/01/2024,2023,NaN
3,01/01/2023,31/12/2023,2023,2023,163083,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,"8206727,38","6243403,3","6243403,3","4561508,41","4561508,41","1963324,08",23/01/2024,2023,NaN
4,01/01/2023,31/12/2023,2023,2023,163084,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,10000000,"8756657,96","8756657,96","8756657,96","6378324,65","1243342,04",23/01/2024,2023,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26572,01/01/2025,31/12/2025,2025,2025,181532,Legislativo,77,FTCMSP,Fundo Especial de Despesas do Tribunal de Contas,10,...,0,660000,0,0,0,0,660000,16/01/2026,2025,2017 - Escola de Gestão e Contas Públicas
26573,01/01/2025,31/12/2025,2025,2025,181534,Legislativo,77,FTCMSP,Fundo Especial de Despesas do Tribunal de Contas,10,...,0,635000,"100819,63","100819,63","100220,06","100220,06","534180,37",16/01/2026,2025,2017 - Escola de Gestão e Contas Públicas
26574,01/01/2025,31/12/2025,2025,2025,181535,Legislativo,77,FTCMSP,Fundo Especial de Despesas do Tribunal de Contas,10,...,0,132000,0,0,0,0,132000,16/01/2026,2025,2017 - Escola de Gestão e Contas Públicas
26575,01/01/2025,31/12/2025,2025,2025,181543,Legislativo,77,FTCMSP,Fundo Especial de Despesas do Tribunal de Contas,10,...,0,215000,0,0,0,0,215000,16/01/2026,2025,2017 - Escola de Gestão e Contas Públicas


Primeiro, vamos filtrar os dados para manter apenas a função Saúde do orçamento.

In [23]:
df_despesa = df_despesa[df_despesa['Ds_Funcao']=='Saúde']
df_despesa = df_despesa[df_despesa['Cd_AnoExecucao']==df_despesa['ANO']]
df_despesa

,DataInicial,DataFinal,Cd_Exercicio,Cd_AnoExecucao,Cd_Dotacao_Id,Administracao,Cd_Orgao,Sigla_Orgao,Ds_Orgao,Cd_Unidade,...,Vl_CongeladoLiquido,Disponivel,Vl_ReservadoLiquido,Vl_EmpenhadoLiquido,Vl_Liquidado,Vl_Pago,Saldo_Dotacao,DataExtracao,ANO,ProjAtiv_Descricao
3,01/01/2023,31/12/2023,2023,2023,163083,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,"8206727,38","6243403,3","6243403,3","4561508,41","4561508,41","1963324,08",23/01/2024,2023,NaN
4,01/01/2023,31/12/2023,2023,2023,163084,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,10000000,"8756657,96","8756657,96","8756657,96","6378324,65","1243342,04",23/01/2024,2023,NaN
5,01/01/2023,31/12/2023,2023,2023,163086,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,10000000,"3870279,58","3870279,58","3870279,58","2890236,45","6129720,42",23/01/2024,2023,NaN
6,01/01/2023,31/12/2023,2023,2023,159819,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,0,0,0,0,0,0,23/01/2024,2023,NaN
7,01/01/2023,31/12/2023,2023,2023,163085,Direta,07,FMD,Fundo Municipal de Desenvolvimento Social,10,...,0,"55667612,83","25971315,21","25971315,21","2460055,22","2291542,01","29696297,62",23/01/2024,2023,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25889,01/01/2025,31/12/2025,2025,2025,179823,Indireta,02,HSPM,Hospital do Servidor Público Municipal,10,...,0,"298205,4","298205,4","298205,4","294415,4","294415,4",0,16/01/2026,2025,2507 - Manutenção e Operação em Atenção Hospit...
25890,01/01/2025,31/12/2025,2025,2025,192458,Indireta,02,HSPM,Hospital do Servidor Público Municipal,10,...,0,0,0,0,0,0,0,16/01/2026,2025,2507 - Manutenção e Operação em Atenção Hospit...
25891,01/01/2025,31/12/2025,2025,2025,179824,Indireta,02,HSPM,Hospital do Servidor Público Municipal,10,...,0,"2561786,39","1359427,45","1359427,45","591187,8","591187,8","1202358,94",16/01/2026,2025,2507 - Manutenção e Operação em Atenção Hospit...
25892,01/01/2025,31/12/2025,2025,2025,183242,Indireta,02,HSPM,Hospital do Servidor Público Municipal,10,...,0,0,0,0,0,0,0,16/01/2026,2025,"9005 - Reforma, Requalificação e Aquisição de ..."


Agora, agrupamos os dados pelas colunas que serão mantidas nos dados finais.

In [24]:
group_cols = ['Cd_SubFuncao',
        'Ds_SubFuncao',
        'Cd_Programa',
        'Ds_Programa',
        'ANO']

measure_cols = ['Vl_Liquidado']

df_despesa.loc[:, measure_cols] = df_despesa.loc[:, measure_cols].apply(lambda x: x.str.replace('.', ''))
df_despesa.loc[:, measure_cols] = df_despesa.loc[:, measure_cols].apply(lambda x: x.str.replace(',', '.'))
df_despesa.loc[:, measure_cols] = df_despesa.loc[:, measure_cols].apply(pd.to_numeric)

df_despesa_agrupado = df_despesa.groupby(group_cols).sum().round(2).reset_index().loc[:, group_cols + measure_cols]
df_despesa_agrupado

,Cd_SubFuncao,Ds_SubFuncao,Cd_Programa,Ds_Programa,ANO,Vl_Liquidado
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2023,15627927.4
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2024,11147382.3
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2025,831865.25
3,122,Administração Geral,3005,Promoção da Sustentabilidade Ambiental,2024,0.0
4,122,Administração Geral,3011,"Modernização Tecnológica, Desburocratização e ...",2023,5392435.42
5,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2023,0.0
6,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2024,173949.1
7,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2025,150748.11
8,122,Administração Geral,3024,Suporte Administrativo,2023,2594339269.469999
9,122,Administração Geral,3024,Suporte Administrativo,2024,2695751169.82


Baixados todos os dados, vamos começar as transformações para adequar as categorias dos dados às necessidades dos indicadores.

# Transformação e padronização

Primeiro, vamos criar as variáveis que representarão os níveis de atenção padronizados.

In [25]:
NIVEL_COL_NAME = 'Nível de atenção padronizado'
NIVEL_AB_VIGILANCIA = 'Atenção Básica e Vigilância'
NIVEL_ESP = 'Atenção Especializada e Suporte profilático terapêutico'
NIVEL_OUTROS = 'Outros níveis de atenção'

Nas próximas seções, vamos também padronizar os nomes das Subprefeituras, com uma função já disponível no módulo `core.utils.cleaning`.

## Consultas médicas

In [26]:
df_consultas_ajustado

,Subprefeitura,Categoria,Subcategoria,Qtd_Consultas,Ano
0,Aricanduva/Formosa/Carrão,Consulta Médica na Atenção Básica,Não Urgência SMS,188894,2024
1,Butantã,Consulta Médica na Atenção Básica,Não Urgência SMS,443917,2024
2,Campo Limpo,Consulta Médica na Atenção Básica,Não Urgência SMS,837251,2024
3,Capela do Socorro,Consulta Médica na Atenção Básica,Não Urgência SMS,554063,2024
4,Casa Verde/Cachoeirinha,Consulta Médica na Atenção Básica,Não Urgência SMS,327363,2024
...,...,...,...,...,...
187,Sapopemba,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,20983,2025
188,Sé,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,250356,2025
189,Vila Maria/Vila Guilherme,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,0,2025
190,Vila Mariana,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,161286,2025


In [27]:
subpref_map = {
    "Aricanduva/Formosa/Carrão": "ARICANDUVA-FORMOSA-CARRAO",
    "Butantã": "BUTANTA",
    "Campo Limpo": "CAMPO LIMPO",
    "Capela do Socorro": "CAPELA DO SOCORRO",
    "Casa Verde/Cachoeirinha": "CASA VERDE-CACHOEIRINHA",
    "Cidade Ademar": "CIDADE ADEMAR",
    "Cidade Tiradentes": "CIDADE TIRADENTES",
    "Ermelino Matarazzo": "ERMELINO MATARAZZO",
    "Freguesia/Brasilândia": "FREGUESIA-BRASILANDIA",
    "Guaianases": "GUAIANASES",
    "Ipiranga": "IPIRANGA",
    "Itaim Paulista": "ITAIM PAULISTA",
    "Itaquera": "ITAQUERA",
    "Jabaquara": "JABAQUARA",
    "Jaçanã/Tremembé": "JACANA-TREMEMBE",
    "Lapa": "LAPA",
    "M'Boi Mirim": "M BOI MIRIM",
    "Mooca": "MOOCA",
    "Parelheiros": "PARELHEIROS",
    "Penha": "PENHA",
    "Perus": "PERUS",
    "Pinheiros": "PINHEIROS",
    "Pirituba/Jaraguá": "PIRITUBA-JARAGUA",
    "Santana/Tucuruvi": "SANTANA-TUCURUVI",
    "Santo Amaro": "SANTO AMARO",
    "São Mateus": "SAO MATEUS",
    "São Miguel Paulista": "SAO MIGUEL",
    "Sapopemba": "SAPOPEMBA",
    "Sé": "SE",
    "Vila Maria/Vila Guilherme": "VILA MARIA-VILA GUILHERME",
    "Vila Mariana": "VILA MARIANA",
    "Vila Prudente": "VILA PRUDENTE"
}

In [28]:
df_consultas_ajustado.loc[:, 'SUBPREFEITURA_QLIK'] = (
    df_consultas_ajustado.loc[:, 'Subprefeitura'].map(subpref_map)
)

df_consultas_ajustado

,Subprefeitura,Categoria,Subcategoria,Qtd_Consultas,Ano,SUBPREFEITURA_QLIK
0,Aricanduva/Formosa/Carrão,Consulta Médica na Atenção Básica,Não Urgência SMS,188894,2024,ARICANDUVA-FORMOSA-CARRAO
1,Butantã,Consulta Médica na Atenção Básica,Não Urgência SMS,443917,2024,BUTANTA
2,Campo Limpo,Consulta Médica na Atenção Básica,Não Urgência SMS,837251,2024,CAMPO LIMPO
3,Capela do Socorro,Consulta Médica na Atenção Básica,Não Urgência SMS,554063,2024,CAPELA DO SOCORRO
4,Casa Verde/Cachoeirinha,Consulta Médica na Atenção Básica,Não Urgência SMS,327363,2024,CASA VERDE-CACHOEIRINHA
...,...,...,...,...,...,...
187,Sapopemba,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,20983,2025,SAPOPEMBA
188,Sé,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,250356,2025,SE
189,Vila Maria/Vila Guilherme,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,0,2025,VILA MARIA-VILA GUILHERME
190,Vila Mariana,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,161286,2025,VILA MARIANA


In [29]:
df_consultas_ajustado['Categoria'].unique()

array(['Consulta Médica na Atenção Básica',
       'Consulta Médica na Atenção Especializada',
       'Consulta Médica/Atendimento em Urgência/Emergência'], dtype=object)

In [30]:
df_consultas_final = df_consultas_ajustado.copy()

filtro_ab = df_consultas_final['Categoria'].str.contains('Atenção Básica')

df_consultas_final.loc[filtro_ab, NIVEL_COL_NAME] = NIVEL_AB_VIGILANCIA
df_consultas_final.loc[~filtro_ab, NIVEL_COL_NAME] = NIVEL_ESP

df_consultas_final

,Subprefeitura,Categoria,Subcategoria,Qtd_Consultas,Ano,SUBPREFEITURA_QLIK,Nível de atenção padronizado
0,Aricanduva/Formosa/Carrão,Consulta Médica na Atenção Básica,Não Urgência SMS,188894,2024,ARICANDUVA-FORMOSA-CARRAO,Atenção Básica e Vigilância
1,Butantã,Consulta Médica na Atenção Básica,Não Urgência SMS,443917,2024,BUTANTA,Atenção Básica e Vigilância
2,Campo Limpo,Consulta Médica na Atenção Básica,Não Urgência SMS,837251,2024,CAMPO LIMPO,Atenção Básica e Vigilância
3,Capela do Socorro,Consulta Médica na Atenção Básica,Não Urgência SMS,554063,2024,CAPELA DO SOCORRO,Atenção Básica e Vigilância
4,Casa Verde/Cachoeirinha,Consulta Médica na Atenção Básica,Não Urgência SMS,327363,2024,CASA VERDE-CACHOEIRINHA,Atenção Básica e Vigilância
...,...,...,...,...,...,...,...
187,Sapopemba,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,20983,2025,SAPOPEMBA,Atenção Especializada e Suporte profilático te...
188,Sé,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,250356,2025,SE,Atenção Especializada e Suporte profilático te...
189,Vila Maria/Vila Guilherme,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,0,2025,VILA MARIA-VILA GUILHERME,Atenção Especializada e Suporte profilático te...
190,Vila Mariana,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,161286,2025,VILA MARIANA,Atenção Especializada e Suporte profilático te...


In [31]:
df_consultas_final['subprefeitura_nivel_ano'] = (
    df_consultas_final['SUBPREFEITURA_QLIK'] +
    ' | ' +
    df_consultas_final['Nível de atenção padronizado'] +
    ' | ' +
    df_consultas_final['Ano'].astype(str)
)

df_consultas_final

,Subprefeitura,Categoria,Subcategoria,Qtd_Consultas,Ano,SUBPREFEITURA_QLIK,Nível de atenção padronizado,subprefeitura_nivel_ano
0,Aricanduva/Formosa/Carrão,Consulta Médica na Atenção Básica,Não Urgência SMS,188894,2024,ARICANDUVA-FORMOSA-CARRAO,Atenção Básica e Vigilância,ARICANDUVA-FORMOSA-CARRAO | Atenção Básica e V...
1,Butantã,Consulta Médica na Atenção Básica,Não Urgência SMS,443917,2024,BUTANTA,Atenção Básica e Vigilância,BUTANTA | Atenção Básica e Vigilância | 2024
2,Campo Limpo,Consulta Médica na Atenção Básica,Não Urgência SMS,837251,2024,CAMPO LIMPO,Atenção Básica e Vigilância,CAMPO LIMPO | Atenção Básica e Vigilância | 2024
3,Capela do Socorro,Consulta Médica na Atenção Básica,Não Urgência SMS,554063,2024,CAPELA DO SOCORRO,Atenção Básica e Vigilância,CAPELA DO SOCORRO | Atenção Básica e Vigilânci...
4,Casa Verde/Cachoeirinha,Consulta Médica na Atenção Básica,Não Urgência SMS,327363,2024,CASA VERDE-CACHOEIRINHA,Atenção Básica e Vigilância,CASA VERDE-CACHOEIRINHA | Atenção Básica e Vig...
...,...,...,...,...,...,...,...,...
187,Sapopemba,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,20983,2025,SAPOPEMBA,Atenção Especializada e Suporte profilático te...,SAPOPEMBA | Atenção Especializada e Suporte pr...
188,Sé,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,250356,2025,SE,Atenção Especializada e Suporte profilático te...,SE | Atenção Especializada e Suporte profiláti...
189,Vila Maria/Vila Guilherme,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,0,2025,VILA MARIA-VILA GUILHERME,Atenção Especializada e Suporte profilático te...,VILA MARIA-VILA GUILHERME | Atenção Especializ...
190,Vila Mariana,Consulta Médica/Atendimento em Urgência/Emergê...,Gestão SES,161286,2025,VILA MARIANA,Atenção Especializada e Suporte profilático te...,VILA MARIANA | Atenção Especializada e Suporte...


## Leitos de internação e complementares

Primeiro, vamos organizar os leitos nas categorias: Leitos de internações; Leitos complementares.

A regra específicada pelo GT foi a seguinte:

 - Leitos de internações são todas as categorias de leito, exceto as categorias `3-Complementar` e `7-Hospital/DIA`;

 - Leitos complementares são os da categoria `3-Complementar`;

In [32]:
df_leito_ajustado['Categoria específica'].value_counts()

Categoria específica
1-Cirúrgico                32
2-Clínico                  32
3-Complementar             32
4-Obstétrico               32
5-Pediátrico               32
6-Outras Especialidades    32
7-Hospital/DIA             32
Name: count, dtype: int64

In [33]:
leito_map = {
    '1-Cirúrgico': 'Leitos de internação',
    '2-Clínico': 'Leitos de internação',
    '3-Complementar': 'Leitos complementares',
    '4-Obstétrico': 'Leitos de internação',
    '5-Pediátrico': 'Leitos de internação',
    '6-Outras Especialidades': 'Leitos de internação',
    '7-Hospital/DIA': None,
}

df_leito_ajustado.loc[:, 'Categoria geral'] = df_leito_ajustado['Categoria específica'].map(leito_map)
df_leito_ajustado

,Pref Regional,Categoria específica,Qtd leitos,Categoria geral
0,Aricanduva/ Formosa/ Carrão,1-Cirúrgico,0,Leitos de internação
1,Butantã,1-Cirúrgico,78,Leitos de internação
2,Campo Limpo,1-Cirúrgico,0,Leitos de internação
3,Capela do Socorro,1-Cirúrgico,78,Leitos de internação
4,Casa Verde/ Cachoeirinha,1-Cirúrgico,86,Leitos de internação
...,...,...,...,...
219,Sapopemba,7-Hospital/DIA,6,None
220,Sé,7-Hospital/DIA,69,None
221,Vila Maria/ Vila Guilherme,7-Hospital/DIA,6,None
222,Vila Mariana,7-Hospital/DIA,166,None


Agora, adicionamos também a classificação geral por nível de atenção (básica ou especializada). Para leitos, todos são utilizados na Atenção especializada, então é só adicionarmos uma coluna com o valor como padrão.

In [34]:
df_leito_ajustado.loc[:, NIVEL_COL_NAME] = NIVEL_ESP
df_leito_ajustado

,Pref Regional,Categoria específica,Qtd leitos,Categoria geral,Nível de atenção padronizado
0,Aricanduva/ Formosa/ Carrão,1-Cirúrgico,0,Leitos de internação,Atenção Especializada e Suporte profilático te...
1,Butantã,1-Cirúrgico,78,Leitos de internação,Atenção Especializada e Suporte profilático te...
2,Campo Limpo,1-Cirúrgico,0,Leitos de internação,Atenção Especializada e Suporte profilático te...
3,Capela do Socorro,1-Cirúrgico,78,Leitos de internação,Atenção Especializada e Suporte profilático te...
4,Casa Verde/ Cachoeirinha,1-Cirúrgico,86,Leitos de internação,Atenção Especializada e Suporte profilático te...
...,...,...,...,...,...
219,Sapopemba,7-Hospital/DIA,6,None,Atenção Especializada e Suporte profilático te...
220,Sé,7-Hospital/DIA,69,None,Atenção Especializada e Suporte profilático te...
221,Vila Maria/ Vila Guilherme,7-Hospital/DIA,6,None,Atenção Especializada e Suporte profilático te...
222,Vila Mariana,7-Hospital/DIA,166,None,Atenção Especializada e Suporte profilático te...


In [35]:
group_cols = ['Pref Regional',
              'Categoria geral',
              NIVEL_COL_NAME]

df_leitos_final = (
    df_leito_ajustado[group_cols + ['Qtd leitos']]
    .groupby(group_cols)
    .sum()
    .reset_index()
)

df_leitos_final

,Pref Regional,Categoria geral,Nível de atenção padronizado,Qtd leitos
0,Aricanduva/ Formosa/ Carrão,Leitos complementares,Atenção Especializada e Suporte profilático te...,0
1,Aricanduva/ Formosa/ Carrão,Leitos de internação,Atenção Especializada e Suporte profilático te...,95
2,Butantã,Leitos complementares,Atenção Especializada e Suporte profilático te...,66
3,Butantã,Leitos de internação,Atenção Especializada e Suporte profilático te...,344
4,Campo Limpo,Leitos complementares,Atenção Especializada e Suporte profilático te...,0
...,...,...,...,...
59,Vila Maria/ Vila Guilherme,Leitos de internação,Atenção Especializada e Suporte profilático te...,144
60,Vila Mariana,Leitos complementares,Atenção Especializada e Suporte profilático te...,324
61,Vila Mariana,Leitos de internação,Atenção Especializada e Suporte profilático te...,1491
62,Vila Prudente,Leitos complementares,Atenção Especializada e Suporte profilático te...,55


In [36]:
df_leitos_final = df_leitos_final.rename(columns={'Pref Regional': 'Subprefeitura'})
df_leitos_final

,Subprefeitura,Categoria geral,Nível de atenção padronizado,Qtd leitos
0,Aricanduva/ Formosa/ Carrão,Leitos complementares,Atenção Especializada e Suporte profilático te...,0
1,Aricanduva/ Formosa/ Carrão,Leitos de internação,Atenção Especializada e Suporte profilático te...,95
2,Butantã,Leitos complementares,Atenção Especializada e Suporte profilático te...,66
3,Butantã,Leitos de internação,Atenção Especializada e Suporte profilático te...,344
4,Campo Limpo,Leitos complementares,Atenção Especializada e Suporte profilático te...,0
...,...,...,...,...
59,Vila Maria/ Vila Guilherme,Leitos de internação,Atenção Especializada e Suporte profilático te...,144
60,Vila Mariana,Leitos complementares,Atenção Especializada e Suporte profilático te...,324
61,Vila Mariana,Leitos de internação,Atenção Especializada e Suporte profilático te...,1491
62,Vila Prudente,Leitos complementares,Atenção Especializada e Suporte profilático te...,55


In [37]:
subpref_map = {
    "Aricanduva/ Formosa/ Carrão": "ARICANDUVA-FORMOSA-CARRAO",
    "Butantã": "BUTANTA",
    "Campo Limpo": "CAMPO LIMPO",
    "Capela do Socorro": "CAPELA DO SOCORRO",
    "Casa Verde/ Cachoeirinha": "CASA VERDE-CACHOEIRINHA",
    "Cidade Ademar": "CIDADE ADEMAR",
    "Cidade Tiradentes": "CIDADE TIRADENTES",
    "Ermelino Matarazzo": "ERMELINO MATARAZZO",
    "Freguesia/ Brasilândia": "FREGUESIA-BRASILANDIA",
    "Guaianases": "GUAIANASES",
    "Ipiranga": "IPIRANGA",
    "Itaim Paulista": "ITAIM PAULISTA",
    "Itaquera": "ITAQUERA",
    "Jabaquara": "JABAQUARA",
    "Jaçanã/ Tremembé": "JACANA-TREMEMBE",
    "Lapa": "LAPA",
    "M Boi Mirim": "M BOI MIRIM",
    "Mooca": "MOOCA",
    "Parelheiros": "PARELHEIROS",
    "Penha": "PENHA",
    "Perus": "PERUS",
    "Pinheiros": "PINHEIROS",
    "Pirituba/ Jaraguá": "PIRITUBA-JARAGUA",
    "Santana/ Tucuruvi": "SANTANA-TUCURUVI",
    "Santo Amaro": "SANTO AMARO",
    "Sapopemba": "SAPOPEMBA",
    "São Mateus": "SAO MATEUS",
    "São Miguel": "SAO MIGUEL",
    "Sé": "SE",
    "Vila Maria/ Vila Guilherme": "VILA MARIA-VILA GUILHERME",
    "Vila Mariana": "VILA MARIANA",
    "Vila Prudente": "VILA PRUDENTE"
}


In [38]:
df_leitos_final.loc[:, 'SUBPREFEITURA_QLIK'] = (
    df_leitos_final.loc[:, 'Subprefeitura'].map(subpref_map)
)

df_leitos_final

,Subprefeitura,Categoria geral,Nível de atenção padronizado,Qtd leitos,SUBPREFEITURA_QLIK
0,Aricanduva/ Formosa/ Carrão,Leitos complementares,Atenção Especializada e Suporte profilático te...,0,ARICANDUVA-FORMOSA-CARRAO
1,Aricanduva/ Formosa/ Carrão,Leitos de internação,Atenção Especializada e Suporte profilático te...,95,ARICANDUVA-FORMOSA-CARRAO
2,Butantã,Leitos complementares,Atenção Especializada e Suporte profilático te...,66,BUTANTA
3,Butantã,Leitos de internação,Atenção Especializada e Suporte profilático te...,344,BUTANTA
4,Campo Limpo,Leitos complementares,Atenção Especializada e Suporte profilático te...,0,CAMPO LIMPO
...,...,...,...,...,...
59,Vila Maria/ Vila Guilherme,Leitos de internação,Atenção Especializada e Suporte profilático te...,144,VILA MARIA-VILA GUILHERME
60,Vila Mariana,Leitos complementares,Atenção Especializada e Suporte profilático te...,324,VILA MARIANA
61,Vila Mariana,Leitos de internação,Atenção Especializada e Suporte profilático te...,1491,VILA MARIANA
62,Vila Prudente,Leitos complementares,Atenção Especializada e Suporte profilático te...,55,VILA PRUDENTE


In [39]:
df_leitos_final['subprefeitura_nivel'] = (
    df_leitos_final['SUBPREFEITURA_QLIK'] +
    ' | ' +
    df_leitos_final['Nível de atenção padronizado']
)

df_leitos_final

,Subprefeitura,Categoria geral,Nível de atenção padronizado,Qtd leitos,SUBPREFEITURA_QLIK,subprefeitura_nivel
0,Aricanduva/ Formosa/ Carrão,Leitos complementares,Atenção Especializada e Suporte profilático te...,0,ARICANDUVA-FORMOSA-CARRAO,ARICANDUVA-FORMOSA-CARRAO | Atenção Especializ...
1,Aricanduva/ Formosa/ Carrão,Leitos de internação,Atenção Especializada e Suporte profilático te...,95,ARICANDUVA-FORMOSA-CARRAO,ARICANDUVA-FORMOSA-CARRAO | Atenção Especializ...
2,Butantã,Leitos complementares,Atenção Especializada e Suporte profilático te...,66,BUTANTA,BUTANTA | Atenção Especializada e Suporte prof...
3,Butantã,Leitos de internação,Atenção Especializada e Suporte profilático te...,344,BUTANTA,BUTANTA | Atenção Especializada e Suporte prof...
4,Campo Limpo,Leitos complementares,Atenção Especializada e Suporte profilático te...,0,CAMPO LIMPO,CAMPO LIMPO | Atenção Especializada e Suporte ...
...,...,...,...,...,...,...
59,Vila Maria/ Vila Guilherme,Leitos de internação,Atenção Especializada e Suporte profilático te...,144,VILA MARIA-VILA GUILHERME,VILA MARIA-VILA GUILHERME | Atenção Especializ...
60,Vila Mariana,Leitos complementares,Atenção Especializada e Suporte profilático te...,324,VILA MARIANA,VILA MARIANA | Atenção Especializada e Suporte...
61,Vila Mariana,Leitos de internação,Atenção Especializada e Suporte profilático te...,1491,VILA MARIANA,VILA MARIANA | Atenção Especializada e Suporte...
62,Vila Prudente,Leitos complementares,Atenção Especializada e Suporte profilático te...,55,VILA PRUDENTE,VILA PRUDENTE | Atenção Especializada e Suport...


## Despesas

Primeiro, vamos complementar os dados de despesas regionalizados com os não regionalizados. Para isso, precisamos subtrair os dados de despesas detalhados dos dado de despesa não detalhados e verificar quanto dos dados não está na tabela de detalhamento.

In [40]:
df_despesa_r_agg = (
    df_despesa_r_agrupado
    .groupby(['CÓDIGO_SUBFUNÇÃO', 'CÓDIGO_PROGRAMA', 'ANO'])
    .agg({'VALOR_DETALHAMENTO_AÇÃO': 'sum'})
    .round(2)
    .reset_index()
)

df_despesa_ajustado = df_despesa_agrupado.merge(
    df_despesa_r_agg,
    left_on=['Cd_SubFuncao', 'Cd_Programa', 'ANO'],
    right_on=['CÓDIGO_SUBFUNÇÃO', 'CÓDIGO_PROGRAMA', 'ANO'],
    how='left'
).drop(columns=['CÓDIGO_SUBFUNÇÃO', 'CÓDIGO_PROGRAMA'])

df_despesa_ajustado

,Cd_SubFuncao,Ds_SubFuncao,Cd_Programa,Ds_Programa,ANO,Vl_Liquidado,VALOR_DETALHAMENTO_AÇÃO
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2023,15627927.4,15627927.4
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2024,11147382.3,11127381.46
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2025,831865.25,692987.18
3,122,Administração Geral,3005,Promoção da Sustentabilidade Ambiental,2024,0.0,NaN
4,122,Administração Geral,3011,"Modernização Tecnológica, Desburocratização e ...",2023,5392435.42,5392435.42
5,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2023,0.0,NaN
6,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2024,173949.1,173949.1
7,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2025,150748.11,133895.61
8,122,Administração Geral,3024,Suporte Administrativo,2023,2594339269.469999,2136984611.549992
9,122,Administração Geral,3024,Suporte Administrativo,2024,2695751169.82,2016199057.540011


In [41]:
df_despesa_ajustado.loc[:, 'Vl_Liquidado_N_Detalhado'] = (
    df_despesa_ajustado.loc[:, 'Vl_Liquidado']
    - df_despesa_ajustado.loc[:, 'VALOR_DETALHAMENTO_AÇÃO']
).round(2)

df_despesa_ajustado

,Cd_SubFuncao,Ds_SubFuncao,Cd_Programa,Ds_Programa,ANO,Vl_Liquidado,VALOR_DETALHAMENTO_AÇÃO,Vl_Liquidado_N_Detalhado
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2023,15627927.4,15627927.4,-0.0
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2024,11147382.3,11127381.46,20000.84
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2025,831865.25,692987.18,138878.07
3,122,Administração Geral,3005,Promoção da Sustentabilidade Ambiental,2024,0.0,NaN,NaN
4,122,Administração Geral,3011,"Modernização Tecnológica, Desburocratização e ...",2023,5392435.42,5392435.42,0.0
5,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2023,0.0,NaN,NaN
6,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2024,173949.1,173949.1,-0.0
7,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2025,150748.11,133895.61,16852.5
8,122,Administração Geral,3024,Suporte Administrativo,2023,2594339269.469999,2136984611.549992,457354657.920007
9,122,Administração Geral,3024,Suporte Administrativo,2024,2695751169.82,2016199057.540011,679552112.279989


In [42]:
df_despesa_ajustado[df_despesa_ajustado['Vl_Liquidado_N_Detalhado']<0]

,Cd_SubFuncao,Ds_SubFuncao,Cd_Programa,Ds_Programa,ANO,Vl_Liquidado,VALOR_DETALHAMENTO_AÇÃO,Vl_Liquidado_N_Detalhado
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2023,15627927.4,15627927.4,-0.0
6,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2024,173949.1,173949.1,-0.0
31,271,Previdência Básica,3004,Benefícios e Previdência de Funcionários,2024,765658793.37,765658793.41,-0.04


Como todos os valores de liquidação não detalhada são maiores do que zero, não parece haver nenhum problema com as agregação. Então, vamos adicionar os dados não detalhados ao dataframe de orçamento regionalizado.

In [43]:
despesa_cols_map = {'Cd_SubFuncao': 'CÓDIGO_SUBFUNÇÃO',
                      'Ds_SubFuncao': 'DESCRIÇÃO_SUBFUNÇÃO',
                      'Cd_Programa': 'CÓDIGO_PROGRAMA',
                      'Ds_Programa': 'DESCRIÇÃO_PROGRAMA',
                      'Vl_Liquidado_N_Detalhado': 'Vl_Liquidado'}

df_despesa_ajustado = (
    df_despesa_ajustado
    .drop(columns=['VALOR_DETALHAMENTO_AÇÃO', 'Vl_Liquidado'])
    .rename(columns=despesa_cols_map)
    )

df_despesa_ajustado

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,ANO,Vl_Liquidado
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2023,-0.0
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2024,20000.84
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",2025,138878.07
3,122,Administração Geral,3005,Promoção da Sustentabilidade Ambiental,2024,NaN
4,122,Administração Geral,3011,"Modernização Tecnológica, Desburocratização e ...",2023,0.0
5,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2023,NaN
6,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2024,-0.0
7,122,Administração Geral,3012,"Participação, Transparência e Controle Social ...",2025,16852.5
8,122,Administração Geral,3024,Suporte Administrativo,2023,457354657.920007
9,122,Administração Geral,3024,Suporte Administrativo,2024,679552112.279989


In [44]:
df_despesa_r_agrupado = (
    df_despesa_r_agrupado
    .rename(columns={'VALOR_DETALHAMENTO_AÇÃO': 'Vl_Liquidado'})
)

df_despesa_r_agrupado

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,Vl_Liquidado
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,599462.69
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,46700.58
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,22676.79
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,24147.12
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,863.79
...,...,...,...,...,...,...,...,...,...,...
434,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,Oeste,Subprefeitura Pinheiros,Supra-Distrital,Despesa Regionalizável,2023,3244937.22
435,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,Sul,Subprefeitura M'Boi Mirim,Supra-Distrital,Despesa Regionalizável,2025,423326.81
436,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2023,550000.0
437,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2025,13227383.38


In [45]:
df_despesas_final = pd.concat([df_despesa_r_agrupado,
                               df_despesa_ajustado], ignore_index=True)

df_despesas_final

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,Vl_Liquidado
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,599462.69
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,46700.58
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,22676.79
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,24147.12
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,863.79
...,...,...,...,...,...,...,...,...,...,...
494,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2024,4275579.67
495,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2025,401772.94
496,422,"Direitos Individuais, Coletivos e Difusos",3012,"Participação, Transparência e Controle Social ...",NaN,NaN,NaN,NaN,2023,900.0
497,422,"Direitos Individuais, Coletivos e Difusos",3012,"Participação, Transparência e Controle Social ...",NaN,NaN,NaN,NaN,2024,NaN


Finalmente, removemos as linhas sem liquidação.

In [46]:
df_despesas_final = df_despesas_final.loc[~df_despesas_final['Vl_Liquidado'].isna()]
df_despesas_final = df_despesas_final.loc[df_despesas_final['Vl_Liquidado']>0]

df_despesas_final

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,Vl_Liquidado
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,599462.69
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,46700.58
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,22676.79
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,24147.12
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,863.79
...,...,...,...,...,...,...,...,...,...,...
491,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2024,70119901.08
492,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2025,82259401.33
494,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2024,4275579.67
495,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2025,401772.94


Para manter mais flexibilidade, vamos manter todas as colunas presentes na tabela de despesas, mesmo que usemos apenas a coluna de programa. Então, o que vamos fazer aqui é criar a nova coluna de nível de atenção padronizado com base nos programas.

In [47]:
df_despesas_final['DESCRIÇÃO_PROGRAMA'].value_counts()

DESCRIÇÃO_PROGRAMA
Ações e Serviços da Saúde em Atenção Básica, Especialidades e Vigilância      266
Ações e Serviços da Saúde em Atenção Hospitalar e de Urgência e Emergência    127
Suporte Administrativo                                                         34
Ações e Serviços da Saúde Animal                                               20
Modernização Tecnológica, Desburocratização e Inovação do Serviço Público      13
Benefícios e Previdência de Funcionários                                        9
Participação, Transparência e Controle Social da Administração Pública          7
Name: count, dtype: int64

In [48]:
map_despesa = {
    'Ações e Serviços da Saúde em Atenção Básica, Especialidades e Vigilância': NIVEL_AB_VIGILANCIA,
    'Ações e Serviços da Saúde em Atenção Hospitalar e de Urgência e Emergência': NIVEL_ESP
}

df_despesas_final.loc[:, NIVEL_COL_NAME] = df_despesas_final.loc[:, 'DESCRIÇÃO_PROGRAMA'].map(map_despesa)
df_despesas_final.loc[:, NIVEL_COL_NAME] = df_despesas_final.loc[:, NIVEL_COL_NAME].fillna(NIVEL_OUTROS)
df_despesas_final

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,Vl_Liquidado,Nível de atenção padronizado
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,599462.69,Atenção Básica e Vigilância
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,46700.58,Atenção Básica e Vigilância
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,22676.79,Atenção Básica e Vigilância
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,24147.12,Atenção Básica e Vigilância
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,863.79,Atenção Básica e Vigilância
...,...,...,...,...,...,...,...,...,...,...,...
491,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2024,70119901.08,Atenção Básica e Vigilância
492,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2025,82259401.33,Atenção Básica e Vigilância
494,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2024,4275579.67,Outros níveis de atenção
495,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2025,401772.94,Outros níveis de atenção


Vamos também criar uma nova coluna com o nome da subprefeitura, de acordo com o padrão existente no arquivo com os polígonos do Qlik.

In [49]:
subpref_filter = (
    df_despesas_final.loc[:, 'SUBPREFEITURA']
    .str.startswith('Subprefeitura ')
    .fillna(False)
)

df_despesas_final.loc[subpref_filter, 'SUBPREFEITURA_QLIK'] = (
    df_despesas_final
    .loc[subpref_filter, 'SUBPREFEITURA']
    .str.removeprefix('Subprefeitura ')
)

df_despesas_final

/tmp/ipykernel_923/3367335398.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,Vl_Liquidado,Nível de atenção padronizado,SUBPREFEITURA_QLIK
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,599462.69,Atenção Básica e Vigilância,Sé
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,46700.58,Atenção Básica e Vigilância,São Mateus
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,22676.79,Atenção Básica e Vigilância,de Guaianases
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,24147.12,Atenção Básica e Vigilância,Pirituba/Jaraguá
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,863.79,Atenção Básica e Vigilância,Santo Amaro
...,...,...,...,...,...,...,...,...,...,...,...,...
491,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2024,70119901.08,Atenção Básica e Vigilância,NaN
492,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2025,82259401.33,Atenção Básica e Vigilância,NaN
494,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2024,4275579.67,Outros níveis de atenção,NaN
495,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2025,401772.94,Outros níveis de atenção,NaN


In [50]:
subpref_map = {
    'Aricanduva/Formosa/Carrão': 'ARICANDUVA-FORMOSA-CARRAO',
    'Butantã': 'BUTANTA',
    'Campo Limpo': 'CAMPO LIMPO',
    'Capela do Socorro': 'CAPELA DO SOCORRO',
    'Casa Verde/Cachoeirinha': 'CASA VERDE-CACHOEIRINHA',
    'Cidade Ademar': 'CIDADE ADEMAR',
    'Cidade Tiradentes': 'CIDADE TIRADENTES',
    'Ermelino Matarazzo': 'ERMELINO MATARAZZO',
    'Freguesia/Brasilândia': 'FREGUESIA-BRASILANDIA',
    'Ipiranga': 'IPIRANGA',
    'Itaim Paulista': 'ITAIM PAULISTA',
    'Itaquera': 'ITAQUERA',
    'Jabaquara': 'JABAQUARA',
    'Jaçanã/Tremembé': 'JACANA-TREMEMBE',
    'Lapa': 'LAPA',
    "M'Boi Mirim": 'M BOI MIRIM',
    'Mooca': 'MOOCA',
    'Parelheiros': 'PARELHEIROS',
    'Penha': 'PENHA',
    'Perus/Anhanguera': 'PERUS',
    'Pinheiros': 'PINHEIROS',
    'Pirituba/Jaraguá': 'PIRITUBA-JARAGUA',
    'Santana/Tucuruvi': 'SANTANA-TUCURUVI',
    'Santo Amaro': 'SANTO AMARO',
    'Sapopemba': 'SAPOPEMBA',
    'São Mateus': 'SAO MATEUS',
    'São Miguel Paulista': 'SAO MIGUEL',
    'Sé': 'SE',
    'Vila Maria/Vila Guilherme': 'VILA MARIA-VILA GUILHERME',
    'Vila Mariana': 'VILA MARIANA',
    'de Guaianases': 'GUAIANASES',
    'de Vila Prudente': 'VILA PRUDENTE'
}

In [51]:
df_despesas_final.loc[subpref_filter, 'SUBPREFEITURA_QLIK'] = (
    df_despesas_final.loc[subpref_filter, 'SUBPREFEITURA_QLIK'].map(subpref_map)
)

df_despesas_final

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,Vl_Liquidado,Nível de atenção padronizado,SUBPREFEITURA_QLIK
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,599462.69,Atenção Básica e Vigilância,SE
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,46700.58,Atenção Básica e Vigilância,SAO MATEUS
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,22676.79,Atenção Básica e Vigilância,GUAIANASES
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,24147.12,Atenção Básica e Vigilância,PIRITUBA-JARAGUA
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,863.79,Atenção Básica e Vigilância,SANTO AMARO
...,...,...,...,...,...,...,...,...,...,...,...,...
491,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2024,70119901.08,Atenção Básica e Vigilância,NaN
492,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2025,82259401.33,Atenção Básica e Vigilância,NaN
494,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2024,4275579.67,Outros níveis de atenção,NaN
495,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2025,401772.94,Outros níveis de atenção,NaN


In [52]:
df_despesas_final['subprefeitura_nivel_ano'] = (
    df_despesas_final['SUBPREFEITURA_QLIK'] +
    ' | ' +
    df_despesas_final['Nível de atenção padronizado'] +
    ' | ' +
    df_despesas_final['ANO']
)

df_despesas_final

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,Vl_Liquidado,Nível de atenção padronizado,SUBPREFEITURA_QLIK,subprefeitura_nivel_ano
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,599462.69,Atenção Básica e Vigilância,SE,SE | Atenção Básica e Vigilância | 2025
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,46700.58,Atenção Básica e Vigilância,SAO MATEUS,SAO MATEUS | Atenção Básica e Vigilância | 2025
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,22676.79,Atenção Básica e Vigilância,GUAIANASES,GUAIANASES | Atenção Básica e Vigilância | 2025
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,24147.12,Atenção Básica e Vigilância,PIRITUBA-JARAGUA,PIRITUBA-JARAGUA | Atenção Básica e Vigilância...
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,863.79,Atenção Básica e Vigilância,SANTO AMARO,SANTO AMARO | Atenção Básica e Vigilância | 2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...
491,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2024,70119901.08,Atenção Básica e Vigilância,NaN,NaN
492,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2025,82259401.33,Atenção Básica e Vigilância,NaN,NaN
494,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2024,4275579.67,Outros níveis de atenção,NaN,NaN
495,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2025,401772.94,Outros níveis de atenção,NaN,NaN


Por último, vamos criar uma coluna para classificar o nível de regionalização.

In [53]:
df_despesas_final.loc[
    ~df_despesas_final['REGIÃO'].str.contains('Supra', na=False),
    'NIVEL_REGIONALIZAÇÃO'] = 'Região'

df_despesas_final

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,Vl_Liquidado,Nível de atenção padronizado,SUBPREFEITURA_QLIK,subprefeitura_nivel_ano,NIVEL_REGIONALIZAÇÃO
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,599462.69,Atenção Básica e Vigilância,SE,SE | Atenção Básica e Vigilância | 2025,Região
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,46700.58,Atenção Básica e Vigilância,SAO MATEUS,SAO MATEUS | Atenção Básica e Vigilância | 2025,Região
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,22676.79,Atenção Básica e Vigilância,GUAIANASES,GUAIANASES | Atenção Básica e Vigilância | 2025,Região
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,24147.12,Atenção Básica e Vigilância,PIRITUBA-JARAGUA,PIRITUBA-JARAGUA | Atenção Básica e Vigilância...,Região
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,863.79,Atenção Básica e Vigilância,SANTO AMARO,SANTO AMARO | Atenção Básica e Vigilância | 2024,Região
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2024,70119901.08,Atenção Básica e Vigilância,NaN,NaN,Região
492,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2025,82259401.33,Atenção Básica e Vigilância,NaN,NaN,Região
494,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2024,4275579.67,Outros níveis de atenção,NaN,NaN,Região
495,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2025,401772.94,Outros níveis de atenção,NaN,NaN,Região


In [54]:
df_despesas_final.loc[
    ~df_despesas_final['SUBPREFEITURA'].str.contains('Supra', na=False),
    'NIVEL_REGIONALIZAÇÃO'] = 'Subprefeitura'

df_despesas_final

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,Vl_Liquidado,Nível de atenção padronizado,SUBPREFEITURA_QLIK,subprefeitura_nivel_ano,NIVEL_REGIONALIZAÇÃO
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,599462.69,Atenção Básica e Vigilância,SE,SE | Atenção Básica e Vigilância | 2025,Subprefeitura
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,46700.58,Atenção Básica e Vigilância,SAO MATEUS,SAO MATEUS | Atenção Básica e Vigilância | 2025,Subprefeitura
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,22676.79,Atenção Básica e Vigilância,GUAIANASES,GUAIANASES | Atenção Básica e Vigilância | 2025,Subprefeitura
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,24147.12,Atenção Básica e Vigilância,PIRITUBA-JARAGUA,PIRITUBA-JARAGUA | Atenção Básica e Vigilância...,Subprefeitura
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,863.79,Atenção Básica e Vigilância,SANTO AMARO,SANTO AMARO | Atenção Básica e Vigilância | 2024,Subprefeitura
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2024,70119901.08,Atenção Básica e Vigilância,NaN,NaN,Subprefeitura
492,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2025,82259401.33,Atenção Básica e Vigilância,NaN,NaN,Subprefeitura
494,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2024,4275579.67,Outros níveis de atenção,NaN,NaN,Subprefeitura
495,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2025,401772.94,Outros níveis de atenção,NaN,NaN,Subprefeitura


In [55]:
df_despesas_final.loc[df_despesas_final['TIPO_REGIONALIZAÇÃO']=='Despesa Não-Regionalizável', 'NIVEL_REGIONALIZAÇÃO'] = 'Não regionalizável'
df_despesas_final

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,Vl_Liquidado,Nível de atenção padronizado,SUBPREFEITURA_QLIK,subprefeitura_nivel_ano,NIVEL_REGIONALIZAÇÃO
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,599462.69,Atenção Básica e Vigilância,SE,SE | Atenção Básica e Vigilância | 2025,Subprefeitura
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,46700.58,Atenção Básica e Vigilância,SAO MATEUS,SAO MATEUS | Atenção Básica e Vigilância | 2025,Subprefeitura
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,22676.79,Atenção Básica e Vigilância,GUAIANASES,GUAIANASES | Atenção Básica e Vigilância | 2025,Subprefeitura
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,24147.12,Atenção Básica e Vigilância,PIRITUBA-JARAGUA,PIRITUBA-JARAGUA | Atenção Básica e Vigilância...,Subprefeitura
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,863.79,Atenção Básica e Vigilância,SANTO AMARO,SANTO AMARO | Atenção Básica e Vigilância | 2024,Subprefeitura
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2024,70119901.08,Atenção Básica e Vigilância,NaN,NaN,Subprefeitura
492,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2025,82259401.33,Atenção Básica e Vigilância,NaN,NaN,Subprefeitura
494,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2024,4275579.67,Outros níveis de atenção,NaN,NaN,Subprefeitura
495,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2025,401772.94,Outros níveis de atenção,NaN,NaN,Subprefeitura


In [56]:
df_despesas_final.loc[df_despesas_final['NIVEL_REGIONALIZAÇÃO'].isna(), 'NIVEL_REGIONALIZAÇÃO'] = 'Não detalhado'
df_despesas_final

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,Vl_Liquidado,Nível de atenção padronizado,SUBPREFEITURA_QLIK,subprefeitura_nivel_ano,NIVEL_REGIONALIZAÇÃO
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,599462.69,Atenção Básica e Vigilância,SE,SE | Atenção Básica e Vigilância | 2025,Subprefeitura
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,46700.58,Atenção Básica e Vigilância,SAO MATEUS,SAO MATEUS | Atenção Básica e Vigilância | 2025,Subprefeitura
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,22676.79,Atenção Básica e Vigilância,GUAIANASES,GUAIANASES | Atenção Básica e Vigilância | 2025,Subprefeitura
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,24147.12,Atenção Básica e Vigilância,PIRITUBA-JARAGUA,PIRITUBA-JARAGUA | Atenção Básica e Vigilância...,Subprefeitura
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,863.79,Atenção Básica e Vigilância,SANTO AMARO,SANTO AMARO | Atenção Básica e Vigilância | 2024,Subprefeitura
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2024,70119901.08,Atenção Básica e Vigilância,NaN,NaN,Subprefeitura
492,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2025,82259401.33,Atenção Básica e Vigilância,NaN,NaN,Subprefeitura
494,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2024,4275579.67,Outros níveis de atenção,NaN,NaN,Subprefeitura
495,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2025,401772.94,Outros níveis de atenção,NaN,NaN,Subprefeitura


Por fim, redefinimos as colunas de valor como texto, arredondando os valores para duas casas decimais após a vírgula e usando vírgula como separador decimal.

In [57]:
despesas_final_value_cols = [
    'Vl_Liquidado'
]

df_despesas_final['Vl_Liquidado'] = (
    df_despesas_final['Vl_Liquidado']
    .round(2)
    .map('{:.2f}'.format)
    .str.replace('.',',')
)

df_despesas_final

,CÓDIGO_SUBFUNÇÃO,DESCRIÇÃO_SUBFUNÇÃO,CÓDIGO_PROGRAMA,DESCRIÇÃO_PROGRAMA,REGIÃO,SUBPREFEITURA,DISTRITO,TIPO_REGIONALIZAÇÃO,ANO,Vl_Liquidado,Nível de atenção padronizado,SUBPREFEITURA_QLIK,subprefeitura_nivel_ano,NIVEL_REGIONALIZAÇÃO
0,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Centro,Subprefeitura Sé,Supra-Distrital,Despesa Regionalizável,2025,"599462,69",Atenção Básica e Vigilância,SE,SE | Atenção Básica e Vigilância | 2025,Subprefeitura
1,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura São Mateus,Supra-Distrital,Despesa Regionalizável,2025,"46700,58",Atenção Básica e Vigilância,SAO MATEUS,SAO MATEUS | Atenção Básica e Vigilância | 2025,Subprefeitura
2,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Leste,Subprefeitura de Guaianases,Supra-Distrital,Despesa Regionalizável,2025,"22676,79",Atenção Básica e Vigilância,GUAIANASES,GUAIANASES | Atenção Básica e Vigilância | 2025,Subprefeitura
3,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Norte,Subprefeitura Pirituba/Jaraguá,Supra-Distrital,Despesa Regionalizável,2025,"24147,12",Atenção Básica e Vigilância,PIRITUBA-JARAGUA,PIRITUBA-JARAGUA | Atenção Básica e Vigilância...,Subprefeitura
4,122,Administração Geral,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",Sul,Subprefeitura Santo Amaro,Supra-Distrital,Despesa Regionalizável,2024,"863,79",Atenção Básica e Vigilância,SANTO AMARO,SANTO AMARO | Atenção Básica e Vigilância | 2024,Subprefeitura
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
491,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2024,"70119901,08",Atenção Básica e Vigilância,NaN,NaN,Subprefeitura
492,304,Vigilância Sanitária,3003,"Ações e Serviços da Saúde em Atenção Básica, E...",NaN,NaN,NaN,NaN,2025,"82259401,33",Atenção Básica e Vigilância,NaN,NaN,Subprefeitura
494,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2024,"4275579,67",Outros níveis de atenção,NaN,NaN,Subprefeitura
495,304,Vigilância Sanitária,3027,Ações e Serviços da Saúde Animal,NaN,NaN,NaN,NaN,2025,"401772,94",Outros níveis de atenção,NaN,NaN,Subprefeitura


## Mortalidade

In [58]:
subpref_map = {
    'Aricanduva/Formosa/Carrão': 'ARICANDUVA-FORMOSA-CARRAO',
    'Butantã': 'BUTANTA',
    'Campo Limpo': 'CAMPO LIMPO',
    'Capela do Socorro': 'CAPELA DO SOCORRO',
    'Casa Verde/Cachoeirinha': 'CASA VERDE-CACHOEIRINHA',
    'Cidade Ademar': 'CIDADE ADEMAR',
    'Cidade Tiradentes': 'CIDADE TIRADENTES',
    'Ermelino Matarazzo': 'ERMELINO MATARAZZO',
    'Freguesia/Brasilândia': 'FREGUESIA-BRASILANDIA',
    'Guaianases': 'GUAIANASES',
    'Ipiranga': 'IPIRANGA',
    'Itaim Paulista': 'ITAIM PAULISTA',
    'Itaquera': 'ITAQUERA',
    'Jabaquara': 'JABAQUARA',
    'Jaçanã/Tremembé': 'JACANA-TREMEMBE',
    'Lapa': 'LAPA',
    "M'Boi Mirim": 'M BOI MIRIM',
    'Moóca': 'MOOCA',
    'Parelheiros': 'PARELHEIROS',
    'Penha': 'PENHA',
    'Perus': 'PERUS',
    'Pinheiros': 'PINHEIROS',
    'Pirituba': 'PIRITUBA-JARAGUA',
    'Santana/Tucuruvi': 'SANTANA-TUCURUVI',
    'Santo Amaro': 'SANTO AMARO',
    'Sapopemba': 'SAPOPEMBA',
    'São Mateus': 'SAO MATEUS',
    'São Miguel': 'SAO MIGUEL',
    'Sé': 'SE',
    'Vila Maria/Vila Guilherme': 'VILA MARIA-VILA GUILHERME',
    'Vila Mariana': 'VILA MARIANA',
    'Vila Prudente': 'VILA PRUDENTE'
}

In [59]:
df_mortalidade_ajustado.loc[:, 'SUBPREFEITURA_QLIK'] = (
    df_mortalidade_ajustado.loc[:, 'Subprefeitura residência'].map(subpref_map)
)

df_mortalidade_ajustado

,Subprefeitura residência,Causa específica,Qtd óbitos,SUBPREFEITURA_QLIK
0,Aricanduva/Formosa/Carrão,Doencas infec intest,4,ARICANDUVA-FORMOSA-CARRAO
1,Butantã,Doencas infec intest,5,BUTANTA
2,Campo Limpo,Doencas infec intest,7,CAMPO LIMPO
3,Capela do Socorro,Doencas infec intest,7,CAPELA DO SOCORRO
4,Casa Verde/Cachoeirinha,Doencas infec intest,4,CASA VERDE-CACHOEIRINHA
...,...,...,...,...
2139,Sapopemba,Demais causas de morte,279,SAPOPEMBA
2140,Sé,Demais causas de morte,285,SE
2141,Vila Maria/Vila Guilherme,Demais causas de morte,279,VILA MARIA-VILA GUILHERME
2142,Vila Mariana,Demais causas de morte,185,VILA MARIANA


Para classificar as doenças em evitáveis e/ou tratáveis, foi elabora uma planilha complementar com os pesos de cada categoria para as causas específicas disponíveis no tabnet.

Vamos carregar essa planilha e utilizar como fator de ponderação.

In [60]:
data_path = path.join('data', 'input', 'saude')
mortalidade_filename = 'DOENÇAS EVITAVEIS E-OU TRATAVEIS.xlsx'
mortalidade_file_path = path.join(data_path, mortalidade_filename)

df_evitaveis_trataveis = pd.read_excel(mortalidade_file_path, sheet_name=1)

df_evitaveis_trataveis

,Categorias específicas,Evitáveis,Tratáveis
0,Acid trânsito e transporte terrestres,1.0,NaN
1,Aids,1.0,NaN
2,Alcoolismo,1.0,NaN
3,Anemias,1.0,NaN
4,Aneurisma e dissecção aorta,0.5,0.5
...,...,...,...
62,Quedas acidentais,1.0,NaN
63,Septicemia,NaN,1.0
64,Suicídios,0.5,0.0
65,Tuberculose,0.5,0.5


In [61]:
df_evitaveis_trataveis = df_evitaveis_trataveis.fillna(0)
df_evitaveis_trataveis

,Categorias específicas,Evitáveis,Tratáveis
0,Acid trânsito e transporte terrestres,1.0,0.0
1,Aids,1.0,0.0
2,Alcoolismo,1.0,0.0
3,Anemias,1.0,0.0
4,Aneurisma e dissecção aorta,0.5,0.5
...,...,...,...
62,Quedas acidentais,1.0,0.0
63,Septicemia,0.0,1.0
64,Suicídios,0.5,0.0
65,Tuberculose,0.5,0.5


In [62]:
df_mortalidade_ajustado = df_mortalidade_ajustado.merge(df_evitaveis_trataveis,
                              left_on='Causa específica',
                              right_on='Categorias específicas',
                              how='left')

df_mortalidade_ajustado

,Subprefeitura residência,Causa específica,Qtd óbitos,SUBPREFEITURA_QLIK,Categorias específicas,Evitáveis,Tratáveis
0,Aricanduva/Formosa/Carrão,Doencas infec intest,4,ARICANDUVA-FORMOSA-CARRAO,Doencas infec intest,1.0,0.0
1,Butantã,Doencas infec intest,5,BUTANTA,Doencas infec intest,1.0,0.0
2,Campo Limpo,Doencas infec intest,7,CAMPO LIMPO,Doencas infec intest,1.0,0.0
3,Capela do Socorro,Doencas infec intest,7,CAPELA DO SOCORRO,Doencas infec intest,1.0,0.0
4,Casa Verde/Cachoeirinha,Doencas infec intest,4,CASA VERDE-CACHOEIRINHA,Doencas infec intest,1.0,0.0
...,...,...,...,...,...,...,...
2139,Sapopemba,Demais causas de morte,279,SAPOPEMBA,Demais causas de morte,0.0,0.0
2140,Sé,Demais causas de morte,285,SE,Demais causas de morte,0.0,0.0
2141,Vila Maria/Vila Guilherme,Demais causas de morte,279,VILA MARIA-VILA GUILHERME,Demais causas de morte,0.0,0.0
2142,Vila Mariana,Demais causas de morte,185,VILA MARIANA,Demais causas de morte,0.0,0.0


Nessa versão, a planilha não classica a causa específica `Infec menigocóccica`. Por ora, vou atribuir o peso 0 nas duas categorias, mas solicitarei a revisão dessa causa específica aos especialistas do GT Saúde.

In [63]:
df_mortalidade_ajustado.loc[:, 'Evitáveis'] = df_mortalidade_ajustado.loc[:, 'Evitáveis'].fillna(0)
df_mortalidade_ajustado.loc[:, 'Tratáveis'] = df_mortalidade_ajustado.loc[:, 'Tratáveis'].fillna(0)
df_mortalidade_ajustado = df_mortalidade_ajustado\
                            .drop(columns='Categorias específicas')

df_mortalidade_ajustado

,Subprefeitura residência,Causa específica,Qtd óbitos,SUBPREFEITURA_QLIK,Evitáveis,Tratáveis
0,Aricanduva/Formosa/Carrão,Doencas infec intest,4,ARICANDUVA-FORMOSA-CARRAO,1.0,0.0
1,Butantã,Doencas infec intest,5,BUTANTA,1.0,0.0
2,Campo Limpo,Doencas infec intest,7,CAMPO LIMPO,1.0,0.0
3,Capela do Socorro,Doencas infec intest,7,CAPELA DO SOCORRO,1.0,0.0
4,Casa Verde/Cachoeirinha,Doencas infec intest,4,CASA VERDE-CACHOEIRINHA,1.0,0.0
...,...,...,...,...,...,...
2139,Sapopemba,Demais causas de morte,279,SAPOPEMBA,0.0,0.0
2140,Sé,Demais causas de morte,285,SE,0.0,0.0
2141,Vila Maria/Vila Guilherme,Demais causas de morte,279,VILA MARIA-VILA GUILHERME,0.0,0.0
2142,Vila Mariana,Demais causas de morte,185,VILA MARIANA,0.0,0.0


In [64]:
df_mortalidade_ajustado = df_mortalidade_ajustado.rename(
    columns={'Evitáveis': 'Qtd evitáveis', 'Tratáveis': 'Qtd tratáveis'}
)

df_mortalidade_ajustado

,Subprefeitura residência,Causa específica,Qtd óbitos,SUBPREFEITURA_QLIK,Qtd evitáveis,Qtd tratáveis
0,Aricanduva/Formosa/Carrão,Doencas infec intest,4,ARICANDUVA-FORMOSA-CARRAO,1.0,0.0
1,Butantã,Doencas infec intest,5,BUTANTA,1.0,0.0
2,Campo Limpo,Doencas infec intest,7,CAMPO LIMPO,1.0,0.0
3,Capela do Socorro,Doencas infec intest,7,CAPELA DO SOCORRO,1.0,0.0
4,Casa Verde/Cachoeirinha,Doencas infec intest,4,CASA VERDE-CACHOEIRINHA,1.0,0.0
...,...,...,...,...,...,...
2139,Sapopemba,Demais causas de morte,279,SAPOPEMBA,0.0,0.0
2140,Sé,Demais causas de morte,285,SE,0.0,0.0
2141,Vila Maria/Vila Guilherme,Demais causas de morte,279,VILA MARIA-VILA GUILHERME,0.0,0.0
2142,Vila Mariana,Demais causas de morte,185,VILA MARIANA,0.0,0.0


In [65]:
df_mortalidade_ajustado.loc[:, 'Qtd evitáveis'] = (
    df_mortalidade_ajustado.loc[:, 'Qtd evitáveis']*df_mortalidade_ajustado.loc[:, 'Qtd óbitos']
)

df_mortalidade_ajustado.loc[:, 'Qtd tratáveis'] = (
    df_mortalidade_ajustado.loc[:, 'Qtd tratáveis']*df_mortalidade_ajustado.loc[:, 'Qtd óbitos']
)

/tmp/ipykernel_923/2988763396.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[4.0 5.0 7.0 ... 0.0 0.0 0.0]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_mortalidade_ajustado.loc[:, 'Qtd evitáveis'] = (
/tmp/ipykernel_923/2988763396.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.0 0.0 0.0 ... 0.0 0.0 0.0]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_mortalidade_ajustado.loc[:, 'Qtd tratáveis'] = (


In [66]:
df_mortalidade_ajustado

,Subprefeitura residência,Causa específica,Qtd óbitos,SUBPREFEITURA_QLIK,Qtd evitáveis,Qtd tratáveis
0,Aricanduva/Formosa/Carrão,Doencas infec intest,4,ARICANDUVA-FORMOSA-CARRAO,4.0,0.0
1,Butantã,Doencas infec intest,5,BUTANTA,5.0,0.0
2,Campo Limpo,Doencas infec intest,7,CAMPO LIMPO,7.0,0.0
3,Capela do Socorro,Doencas infec intest,7,CAPELA DO SOCORRO,7.0,0.0
4,Casa Verde/Cachoeirinha,Doencas infec intest,4,CASA VERDE-CACHOEIRINHA,4.0,0.0
...,...,...,...,...,...,...
2139,Sapopemba,Demais causas de morte,279,SAPOPEMBA,0.0,0.0
2140,Sé,Demais causas de morte,285,SE,0.0,0.0
2141,Vila Maria/Vila Guilherme,Demais causas de morte,279,VILA MARIA-VILA GUILHERME,0.0,0.0
2142,Vila Mariana,Demais causas de morte,185,VILA MARIANA,0.0,0.0


In [67]:
df_mortalidade_final = (
    df_mortalidade_ajustado
    .drop(columns=['Causa específica', 'Qtd óbitos'])
    .groupby(['Subprefeitura residência', 'SUBPREFEITURA_QLIK'])
    .sum()
    .reset_index()
)

df_mortalidade_final

,Subprefeitura residência,SUBPREFEITURA_QLIK,Qtd evitáveis,Qtd tratáveis
0,Aricanduva/Formosa/Carrão,ARICANDUVA-FORMOSA-CARRAO,385.5,463.5
1,Butantã,BUTANTA,491.0,503.5
2,Campo Limpo,CAMPO LIMPO,789.5,831.5
3,Capela do Socorro,CAPELA DO SOCORRO,859.5,934.0
4,Casa Verde/Cachoeirinha,CASA VERDE-CACHOEIRINHA,471.0,575.0
5,Cidade Ademar,CIDADE ADEMAR,542.0,663.5
6,Cidade Tiradentes,CIDADE TIRADENTES,313.0,357.0
7,Ermelino Matarazzo,ERMELINO MATARAZZO,290.0,363.0
8,Freguesia/Brasilândia,FREGUESIA-BRASILANDIA,639.0,747.0
9,Guaianases,GUAIANASES,393.0,493.5


In [68]:
df_mortalidade_final['Qtd evitáveis'] = (
    df_mortalidade_final['Qtd evitáveis']
    .astype(int)
)
df_mortalidade_final['Qtd tratáveis'] = (
    df_mortalidade_final['Qtd tratáveis']
    .astype(int)
)

df_mortalidade_final

,Subprefeitura residência,SUBPREFEITURA_QLIK,Qtd evitáveis,Qtd tratáveis
0,Aricanduva/Formosa/Carrão,ARICANDUVA-FORMOSA-CARRAO,385,463
1,Butantã,BUTANTA,491,503
2,Campo Limpo,CAMPO LIMPO,789,831
3,Capela do Socorro,CAPELA DO SOCORRO,859,934
4,Casa Verde/Cachoeirinha,CASA VERDE-CACHOEIRINHA,471,575
5,Cidade Ademar,CIDADE ADEMAR,542,663
6,Cidade Tiradentes,CIDADE TIRADENTES,313,357
7,Ermelino Matarazzo,ERMELINO MATARAZZO,290,363
8,Freguesia/Brasilândia,FREGUESIA-BRASILANDIA,639,747
9,Guaianases,GUAIANASES,393,493


In [69]:
df_mortalidade_final = (
    df_mortalidade_final
    .melt(
        id_vars=['Subprefeitura residência', 'SUBPREFEITURA_QLIK'],
        var_name='Evitáveis ou tratáveis',
        value_vars=['Qtd evitáveis', 'Qtd tratáveis'],
        value_name='Qtd óbitos'
    )
)

df_mortalidade_final

,Subprefeitura residência,SUBPREFEITURA_QLIK,Evitáveis ou tratáveis,Qtd óbitos
0,Aricanduva/Formosa/Carrão,ARICANDUVA-FORMOSA-CARRAO,Qtd evitáveis,385
1,Butantã,BUTANTA,Qtd evitáveis,491
2,Campo Limpo,CAMPO LIMPO,Qtd evitáveis,789
3,Capela do Socorro,CAPELA DO SOCORRO,Qtd evitáveis,859
4,Casa Verde/Cachoeirinha,CASA VERDE-CACHOEIRINHA,Qtd evitáveis,471
...,...,...,...,...
59,São Miguel,SAO MIGUEL,Qtd tratáveis,724
60,Sé,SE,Qtd tratáveis,532
61,Vila Maria/Vila Guilherme,VILA MARIA-VILA GUILHERME,Qtd tratáveis,503
62,Vila Mariana,VILA MARIANA,Qtd tratáveis,304


In [70]:
df_mortalidade_final.loc[:, 'Evitáveis ou tratáveis'] = (
    df_mortalidade_final
    .loc[:, 'Evitáveis ou tratáveis']
    .str.removeprefix('Qtd ')
    .str.capitalize()
)

df_mortalidade_final

,Subprefeitura residência,SUBPREFEITURA_QLIK,Evitáveis ou tratáveis,Qtd óbitos
0,Aricanduva/Formosa/Carrão,ARICANDUVA-FORMOSA-CARRAO,Evitáveis,385
1,Butantã,BUTANTA,Evitáveis,491
2,Campo Limpo,CAMPO LIMPO,Evitáveis,789
3,Capela do Socorro,CAPELA DO SOCORRO,Evitáveis,859
4,Casa Verde/Cachoeirinha,CASA VERDE-CACHOEIRINHA,Evitáveis,471
...,...,...,...,...
59,São Miguel,SAO MIGUEL,Tratáveis,724
60,Sé,SE,Tratáveis,532
61,Vila Maria/Vila Guilherme,VILA MARIA-VILA GUILHERME,Tratáveis,503
62,Vila Mariana,VILA MARIANA,Tratáveis,304


## Subprefeitura x Nível de atenção

Para que seja possível relacionar todos os indicadores sem nenhuma conversão nas medidas, precisamos de uma tabela com as dimensões em comum entre todas as tabelas para que seja feita a associação no Qlik. Por isso, vamos criar um dataframe que consistirá no produto cartesiano entre subprefeituras e nível de atenção padronizados.

In [71]:
df_niveis = pd.DataFrame([NIVEL_AB_VIGILANCIA, NIVEL_ESP, NIVEL_OUTROS],
                         columns=[NIVEL_COL_NAME])

df_anos = pd.DataFrame([2023, 2024, 2025], columns=['Ano'])

df_subp_nivel_ano = (
    df_mortalidade_final[['SUBPREFEITURA_QLIK']]
    .drop_duplicates()
    .merge(df_niveis, how='cross')
    .merge(df_anos, how='cross')
)

df_subp_nivel_ano

,SUBPREFEITURA_QLIK,Nível de atenção padronizado,Ano
0,ARICANDUVA-FORMOSA-CARRAO,Atenção Básica e Vigilância,2023
1,ARICANDUVA-FORMOSA-CARRAO,Atenção Básica e Vigilância,2024
2,ARICANDUVA-FORMOSA-CARRAO,Atenção Básica e Vigilância,2025
3,ARICANDUVA-FORMOSA-CARRAO,Atenção Especializada e Suporte profilático te...,2023
4,ARICANDUVA-FORMOSA-CARRAO,Atenção Especializada e Suporte profilático te...,2024
...,...,...,...
283,VILA PRUDENTE,Atenção Especializada e Suporte profilático te...,2024
284,VILA PRUDENTE,Atenção Especializada e Suporte profilático te...,2025
285,VILA PRUDENTE,Outros níveis de atenção,2023
286,VILA PRUDENTE,Outros níveis de atenção,2024


In [72]:
df_subp_nivel_ano['subprefeitura_nivel_ano'] = (
    df_subp_nivel_ano['SUBPREFEITURA_QLIK'] +
    ' | ' +
    df_subp_nivel_ano['Nível de atenção padronizado'] +
    ' | ' +
    df_subp_nivel_ano['Ano'].astype(str)    
)

df_subp_nivel_ano

,SUBPREFEITURA_QLIK,Nível de atenção padronizado,Ano,subprefeitura_nivel_ano
0,ARICANDUVA-FORMOSA-CARRAO,Atenção Básica e Vigilância,2023,ARICANDUVA-FORMOSA-CARRAO | Atenção Básica e V...
1,ARICANDUVA-FORMOSA-CARRAO,Atenção Básica e Vigilância,2024,ARICANDUVA-FORMOSA-CARRAO | Atenção Básica e V...
2,ARICANDUVA-FORMOSA-CARRAO,Atenção Básica e Vigilância,2025,ARICANDUVA-FORMOSA-CARRAO | Atenção Básica e V...
3,ARICANDUVA-FORMOSA-CARRAO,Atenção Especializada e Suporte profilático te...,2023,ARICANDUVA-FORMOSA-CARRAO | Atenção Especializ...
4,ARICANDUVA-FORMOSA-CARRAO,Atenção Especializada e Suporte profilático te...,2024,ARICANDUVA-FORMOSA-CARRAO | Atenção Especializ...
...,...,...,...,...
283,VILA PRUDENTE,Atenção Especializada e Suporte profilático te...,2024,VILA PRUDENTE | Atenção Especializada e Suport...
284,VILA PRUDENTE,Atenção Especializada e Suporte profilático te...,2025,VILA PRUDENTE | Atenção Especializada e Suport...
285,VILA PRUDENTE,Outros níveis de atenção,2023,VILA PRUDENTE | Outros níveis de atenção | 2023
286,VILA PRUDENTE,Outros níveis de atenção,2024,VILA PRUDENTE | Outros níveis de atenção | 2024


# Armazenamento dos dados

Finalmente, salvamos os arquivos como csv para utilizarmos no Qlik Sense.

In [73]:
base_path = path.join('data_output', 'saude')

if not path.exists(base_path):
    makedirs(base_path)

for name, df in [('consultas', df_consultas_final),
                 ('leitos-internacao-complementares', df_leitos_final),
                 ('despesas', df_despesas_final),
                 ('mortalidade', df_mortalidade_final),
                 ('subprefeitura-nivel-ano', df_subp_nivel_ano)]:
    
    filepath = path.join(base_path, f'{name}.csv')

    df.to_csv(filepath,
              index=False,
              sep=';',
              decimal=',')